In [1]:
# ============================================================
# DAY 15 — Portfolio Optimizer
# Cell 1 : Imports, paths, constants
#
# Design confirmed by 15-day model portfolio test:
#   Equal allocation beats ATR-weighted by 0.28%
#   ML Bull Cont continuity = 100% accurate exit signal
#   EMA50% = #1 differentiator (winners +17.6%, losers +1.1%)
#   F25d useful but high variance — 4% tiebreaker only
#   Dynamic sector cap — concentrate where strength is
#   MIXED health = valid entry (14/23 stocks are MIXED)
#   RETREATING only = exclude (2/23 stocks)
# ============================================================

import os
import pickle
import pandas as pd
import numpy as np
from datetime import datetime

# ── DATES ────────────────────────────────────────────────────
today_str  = datetime.now().strftime('%Y-%m-%d')
today_file = datetime.now().strftime('%Y%m%d')

# ── BASE PATHS ───────────────────────────────────────────────
BASE_DIR       = r'E:\learn\Project 1 AI Screener\stock-ai-india'
DATA_DIR       = os.path.join(BASE_DIR, 'data')
UNIVERSE_DIR   = os.path.join(DATA_DIR, 'universe')
FUND_DIR       = os.path.join(DATA_DIR, 'fundamentals')
SCORES_DIR     = os.path.join(DATA_DIR, 'scores')
PRICES_DIR     = os.path.join(DATA_DIR, 'prices')
PORTFOLIO_DIR  = os.path.join(DATA_DIR, 'portfolio')
OPT_STATE_DIR  = os.path.join(PORTFOLIO_DIR, 'optimizer')
OPT_MODEL_DIR  = os.path.join(PORTFOLIO_DIR, 'optimizer', 'model_portfolios')
OPT_TESTER_DIR = os.path.join(PORTFOLIO_DIR, 'optimizer', 'testers')
REPORT_DIR     = os.path.join(DATA_DIR, 'reports', 'optimizer')

for d in [OPT_STATE_DIR, OPT_MODEL_DIR, OPT_TESTER_DIR, REPORT_DIR]:
    os.makedirs(d, exist_ok=True)

# ── FILE PATHS ───────────────────────────────────────────────
TECH_FILE      = os.path.join(SCORES_DIR,    'technical_report_full.csv')
LT_PORT_FILE   = os.path.join(PORTFOLIO_DIR, 'long_term_portfolio.csv')
INDICATOR_FILE = os.path.join(PRICES_DIR,    'indicator_data_full.pkl')
STATE_FILE     = os.path.join(OPT_STATE_DIR, 'optimizer_state.csv')

# ── LAYER 1 : SELECTION FILTERS ──────────────────────────────
# Hard gates — ALL must pass.
# Confirmed by 15-day test:
#   FORCEMOT  EMA50%=+1.8% → gate 3 blocks → avoided -11.8%
#   ORICONENT EMA50%=+0.7% → gate 3 blocks → avoided -4.8%
#   ZEAL      Sideways     → gate 5 blocks → avoided -4.2%
MIN_ML_CONF    = 73.0    # ML_Confidence minimum %
MIN_EMA50_PCT  = 5.0     # price must be ≥ 5% above EMA50
MIN_TECH_SCORE = 75      # Tech Score minimum
# Tier is NO LONGER a hard gate — moved to Layer 2 score bonus
# Removing Tier hard gate: 25 → 53 qualifying stocks
# Tier 1 still preferred — scores higher in Layer 2

# Exact strings from run_weekly.py derive_trend_label()
VALID_SECTOR_TRENDS = ['Strong Uptrend ↑↑', 'Uptrend ↑']

# ── LAYER 2 : SCORING ─────────────────────────────────────────
# Sector trend multiplier
SECTOR_TREND_MULT = {
    'Strong Uptrend ↑↑' : 1.30,
    'Uptrend ↑'          : 1.00,
    'Weak Uptrend →↑'    : 0.70,   # flag engine only
    'Sideways →'         : 0.00,
    'Weak Downtrend →↓'  : 0.00,
    'Strong Downtrend ↓↓': 0.00,
    'Downtrend ↓'        : 0.00,
}

# Tier multiplier — preference not exclusion
# Tier 1 gets full score, Tier 2 gets 90%
# Natural result: Tier 1 fills top slots, Tier 2 fills gaps
TIER1_MULT = 1.00
TIER2_MULT = 0.90
# S6 multiplier — 20-day range sweet spot bonus
# Stocks at 40-75% of 20d range get this bonus in Layer 2 score
# Rationale: fresh bounce from monthly lows = more upside room
# 8% chosen to surface these stocks without distorting formula
S6_SCORE_MULT = 1.08
# Tier 3 still hard-blocked in Gate 6 (weak fundamentals)

# Score weights — confirmed by 15-day test
# EMA50% = #1 differentiator, F25d = tiebreaker only
# Must sum to 1.0
W_EMA50  = 0.33
W_TECH   = 0.23
W_MLCONF = 0.20
W_RANKS  = 0.20
W_F25D   = 0.04

assert abs(W_EMA50 + W_TECH + W_MLCONF + W_RANKS + W_F25D - 1.0) < 1e-9, \
    "Weights must sum to 1.0"

# ── LAYER 3 : POSITION SIZING ─────────────────────────────────
# Equal allocation confirmed by 15-day test:
#   Equal vs rank-weighted = 0.28% difference
#   Stock selection matters 30× more
#
# Formula: capital / n_stocks chosen by user
# Caps applied after:
#   Cap A: per stock  max 15%
#   Cap B: per sector dynamic (max 60%)
#   Cap C: total never exceed 100%
#
# Capital asked AFTER showing qualifying stock list
# User chooses how many stocks (default 10)
MAX_STOCK_PCT       = 0.15   # Cap A: 15% per stock hard ceiling
MAX_SECTOR_ABSOLUTE = 0.60   # Cap B: 60% per sector hard ceiling
DEFAULT_STOCK_COUNT = 10     # default suggestion to user
MAX_STOCK_COUNT     = 20     # hard ceiling on portfolio size
MIN_POSITION_RS     = 15_000 # warn if position < Rs 15,000
# Capital prompted at runtime — never hardcoded

# ATR stop — reference only, not used for sizing or exit trigger
ATR_MULT_STR_UP  = 2.0
ATR_MULT_UPTREND = 1.5

# ── LAYER 4 : MOMENTUM HEALTH ─────────────────────────────────
# 5-signal health check using last 10 candles
# Score -4 to +5
#
# CONFIRMED from diagnostic run (Apr 30 data):
#   23 stocks passed Layer 1
#   7  RISING     → ✅ clean entry
#   14 MIXED      → ✅ valid entry (EMA20 rising, Higher Low confirmed)
#   2  RETREATING → 🚫 exclude (PARAS, SIEMENS)
#
# KEY FIX: MIXED = valid entry, NOT excluded
# MIXED means some short-term signals diverging
# but EMA20 trend and Higher Low still intact
# Excluding MIXED was incorrectly leaving 14 good stocks out
#
# Allocation rules:
#   RISING      → full allocation
#   MIXED       → full allocation (with caution label)
#   NEUTRAL     → full allocation (consolidating, valid)
#   RETREATING  → excluded from auto-tester
#                 shown with warning in manual mode

HEALTH_AUTO_EXIT_THRESHOLD = -3   # auto-tester exits at this score
HEALTH_EXCLUDE_THRESHOLD   = -1   # auto-tester excludes at entry
# Manual mode: user sees all and decides

# Re-entry after exit
REENTRY_GAP_DAYS = 3   # trading days before same stock re-eligible

# Corporate action guard
CORP_ACTION_DROP_PCT   = 20.0
CORP_ACTION_MARKET_PCT = 5.0

# ── EXIT RULES ────────────────────────────────────────────────
# Confirmed 100% accuracy in 15-day test:
#   All Bull Cont maintained → gained 16-54%
#   All Bull Cont degraded   → lost 4-12%
EXIT_FLAG_LABELS = [
    'Bullish', 'Bearish Continual', 'Bearish',
    'No Signal', 'Tech Bullish'
]

# Flag thresholds
MONITOR_EMA50_PCT = 5.0
TREND_ORDER = [
    'Strong Uptrend ↑↑',
    'Uptrend ↑',
    'Weak Uptrend →↑',
    'Sideways →',
    'Weak Downtrend →↓',
    'Strong Downtrend ↓↓',
    'Downtrend ↓',
]
UPGRADE_ML_TARGET    = 'Bullish Continual'
UPGRADE_TREND_TARGET = 'Strong Uptrend ↑↑'

# ── PORTFOLIO COLUMN DEFINITIONS ─────────────────────────────
OPT_PORT_COLS = [
    'Symbol', 'Entry_Price', 'Entry_Date', 'Quantity',
    'Cap_Category', 'Invested',
    'Sector_Rank_At_Entry', 'Cap_Rank_At_Entry',
    'ML_At_Entry', 'Sector_Trend_At_Entry',
    'EMA50_Pct_At_Entry', 'Optimizer_Score_At_Entry',
    'Health_At_Entry', 'Stop_Price_Ref', 'Notes'
]

OPT_MODEL_COLS = [
    'Symbol', 'Entry_Price', 'Entry_Date', 'Quantity',
    'Cap_Category', 'Invested',
    'Sector_Rank_At_Entry', 'Cap_Rank_At_Entry',
    'ML_At_Entry', 'Sector_Trend_At_Entry',
    'EMA50_Pct_At_Entry', 'Optimizer_Score_At_Entry',
    'Health_At_Entry', 'Stop_Price_Ref',
    'Portfolio_Name', 'Notes'
]

TESTER_PORT_COLS = [
    'Symbol', 'Entry_Price', 'Entry_Date', 'Entry_Time',
    'Quantity', 'Cap_Category', 'Invested',
    'ML_At_Entry', 'Sector_Trend_At_Entry',
    'EMA50_Pct_At_Entry', 'Optimizer_Score_At_Entry',
    'Health_At_Entry', 'Stop_Price_Ref',
    'Last_Checked', 'Status', 'Tester_Name'
]

TESTER_LOG_COLS = [
    'Date', 'Time', 'Action', 'Symbol',
    'Price', 'Quantity', 'Amount',
    'ML_Label', 'Health_Score', 'Health_Label',
    'Sector_Trend', 'EMA50_Pct', 'Optimizer_Score',
    'Reason', 'Portfolio_Value_After', 'Tester_Name'
]

# ── MAIN MENU ─────────────────────────────────────────────────
MENU = """
  ╔════════════════════════════════════════════════════╗
  ║       AI Stock Screener — Portfolio Optimizer      ║
  ╠════════════════════════════════════════════════════╣
  ║                                                    ║
  ║   1. Fresh recommendations                         ║
  ║      Full 4-layer pipeline → ranked list           ║
  ║      → export to model or actual portfolio         ║
  ║                                                    ║
  ║   MODEL PORTFOLIO  (manual test)                   ║
  ║   2. Create / manage model portfolio               ║
  ║   3. Review model portfolio                        ║
  ║      P&L + flags + Nifty comparison + chart        ║
  ║      + new candidates not yet in portfolio         ║
  ║                                                    ║
  ║   AUTO-TESTER  (autonomous paper trading)          ║
  ║   4. Run auto-tester                               ║
  ║      Enters and exits automatically based on       ║
  ║      optimizer criteria. Full log maintained.      ║
  ║                                                    ║
  ║   ACTUAL PORTFOLIO  (real money)                   ║
  ║   5. Build / update actual portfolio               ║
  ║   6. Review actual portfolio                       ║
  ║      P&L + flags + Nifty comparison + chart        ║
  ║      + new candidates not yet in portfolio         ║
  ║                                                    ║
  ║   7. Exit                                          ║
  ║                                                    ║
  ╚════════════════════════════════════════════════════╝
"""

# ── DISPLAY HELPERS ──────────────────────────────────────────
CAP_SHORT = {
    'Large Cap'      : 'L',
    'Mini Large Cap' : 'ML',
    'Mid Cap'        : 'M',
    'Small Cap'      : 'S',
}

def mcap_str(val):
    try:
        v = float(val or 0)
    except:
        return '—'
    if v >= 100000: return f'Rs{v/100000:.1f}L Cr'
    if v >= 1000:   return f'Rs{v/1000:.0f}K Cr'
    return f'Rs{v:.0f}Cr'

def short_trend(trend):
    return {
        'Strong Uptrend ↑↑'  : '✅ Str Up ↑↑',
        'Uptrend ↑'           : '✅ Uptrend ↑',
        'Weak Uptrend →↑'     : '〰️ Wk Up →↑',
        'Sideways →'          : '➡️ Sideways →',
        'Weak Downtrend →↓'   : '⚠️ Wk Dn →↓',
        'Strong Downtrend ↓↓' : '🔴 Str Dn ↓↓',
        'Downtrend ↓'         : '🔴 Downtrend ↓',
    }.get(str(trend), str(trend))

def ask_capital(label='portfolio', default=500_000):
    try:
        raw = input(
            f"\n  Capital to deploy for {label} (Rs) "
            f"[default {default:,.0f}]: "
        ).strip().replace(',', '')
        val = float(raw) if raw else float(default)
        if val <= 0:
            raise ValueError
        print(f"  Capital set to Rs{val:,.0f}")
        return val
    except:
        print(f"  Invalid — using Rs{default:,.0f}")
        return float(default)

def ask_stock_count(max_available, default=None):
    """Ask how many stocks to invest in."""
    default = default or min(DEFAULT_STOCK_COUNT, max_available)
    try:
        raw = input(
            f"\n  How many stocks to invest in? "
            f"[default {default}, max {min(max_available, MAX_STOCK_COUNT)}]: "
        ).strip()
        val = int(raw) if raw else default
        val = max(1, min(val, max_available, MAX_STOCK_COUNT))
        print(f"  Investing in top {val} stocks")
        return val
    except:
        print(f"  Invalid — using {default}")
        return default

# ── VALIDATION ───────────────────────────────────────────────
missing = [f for f in [TECH_FILE, INDICATOR_FILE]
           if not os.path.exists(f)]

if missing:
    print("❌  Missing required files — run run_weekly.py first:")
    for f in missing:
        print(f"     {f}")
else:
    state_status = (
        f"exists ({os.path.getsize(STATE_FILE):,} bytes) "
        f"— will compare vs last run"
        if os.path.exists(STATE_FILE)
        else "not found — first run"
    )
    print("✅  Cell 1 complete")
    print(f"     Base dir       : {BASE_DIR}")
    print(f"     Tech file      : {os.path.basename(TECH_FILE)}")
    print(f"     State file     : {state_status}")
    print(f"     Report dir     : {REPORT_DIR}")
    print()
    print(f"     Selection filters  (Layer 1 — 5 hard gates)")
    print(f"       Min ML conf  : {MIN_ML_CONF}%")
    print(f"       Min EMA50%   : {MIN_EMA50_PCT}%")
    print(f"       Min Tech     : {MIN_TECH_SCORE}")
    print(f"       Tier         : Tier 3 excluded, "
          f"Tier 1/2 both qualify")
    print(f"       Sectors      : {', '.join(VALID_SECTOR_TRENDS)}")
    print()
    print(f"     Position sizing    (Layer 3)")
    print(f"       Capital      : prompted after showing stock list")
    print(f"       Count        : user chooses "
          f"(default {DEFAULT_STOCK_COUNT}, max {MAX_STOCK_COUNT})")
    print(f"       Method       : equal allocation "
          f"(capital ÷ chosen count)")
    print(f"       Per-stock cap: {MAX_STOCK_PCT*100:.0f}% hard ceiling")
    print(f"       Sector cap   : dynamic max {MAX_SECTOR_ABSOLUTE*100:.0f}%")
    print(f"       ATR stop     : reference only")
    print()
    print(f"     Score weights      (Layer 2)")
    print(f"       EMA50%  : {W_EMA50}  ← #1 signal")
    print(f"       Tech    : {W_TECH}")
    print(f"       ML Conf : {W_MLCONF}")
    print(f"       Ranks   : {W_RANKS}")
    print(f"       F25d    : {W_F25D}  ← tiebreaker only")
    print(f"       Tier 1  : {TIER1_MULT}× multiplier")
    print(f"       Tier 2  : {TIER2_MULT}× multiplier")
    print()
    print(f"     Health labels      (Layer 4)")
    print(f"       RISING    → ✅ full allocation")
    print(f"       MIXED     → ✅ full allocation "
          f"(14/23 stocks today — must include)")
    print(f"       NEUTRAL   → ✅ full allocation")
    print(f"       RETREATING → 🚫 excluded "
          f"(PARAS, SIEMENS today — correctly blocked)")
    print()
    print(f"     Exit rules")
    print(f"       Primary  : ML degrades Bull Cont → EXIT")
    print(f"       Secondary: Layer 4 score ≤ "
          f"{HEALTH_AUTO_EXIT_THRESHOLD} → EXIT")
    print(f"       Re-entry : {REENTRY_GAP_DAYS} trading days gap")

✅  Cell 1 complete
     Base dir       : E:\learn\Project 1 AI Screener\stock-ai-india
     Tech file      : technical_report_full.csv
     State file     : exists (3,992 bytes) — will compare vs last run
     Report dir     : E:\learn\Project 1 AI Screener\stock-ai-india\data\reports\optimizer

     Selection filters  (Layer 1 — 5 hard gates)
       Min ML conf  : 73.0%
       Min EMA50%   : 5.0%
       Min Tech     : 75
       Tier         : Tier 3 excluded, Tier 1/2 both qualify
       Sectors      : Strong Uptrend ↑↑, Uptrend ↑

     Position sizing    (Layer 3)
       Capital      : prompted after showing stock list
       Count        : user chooses (default 10, max 20)
       Method       : equal allocation (capital ÷ chosen count)
       Per-stock cap: 15% hard ceiling
       Sector cap   : dynamic max 60%
       ATR stop     : reference only

     Score weights      (Layer 2)
       EMA50%  : 0.33  ← #1 signal
       Tech    : 0.23
       ML Conf : 0.2
       Ranks   : 0.2
   

In [2]:
# ============================================================
# DAY 15 — Portfolio Optimizer
# Cell 2 : Load data + all helpers + model portfolio
#           management + menu loop
#
# Run order: Cell 1 → Cell 2 → Cell 3 → Cell 4
#            then call run_optimizer_menu()
# ============================================================

import yfinance as yf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import time as _time_mod

# ── LOAD CORE DATA ────────────────────────────────────────────
print("Loading data...")

tech_df = pd.read_csv(TECH_FILE)

atr_map        = {}
indicator_data = {}
try:
    with open(INDICATOR_FILE, 'rb') as f:
        indicator_data = pickle.load(f)
    for sym, df_ind in indicator_data.items():
        try:
            row   = df_ind.iloc[-1]
            price = float(row['Close'])
            atr   = float(row['ATR'])
            atr_map[sym] = {
                'ATR'    : round(atr, 2),
                'ATR_pct': round(atr / price * 100, 2)
                            if price > 0 else 0
            }
        except:
            pass
    print(f"  Tech data    : {len(tech_df)} stocks")
    print(f"  ATR data     : {len(atr_map)} stocks")
except Exception as e:
    print(f"  ⚠️  indicator_data load failed: {e}")

# Load optimizer state from last run
prev_state = None
if os.path.exists(STATE_FILE):
    try:
        prev_state = pd.read_csv(STATE_FILE)
        print(f"  State file   : {len(prev_state)} rows "
              f"— comparing vs last run")
    except:
        print("  ⚠️  State file unreadable — first run")

# Show data freshness
tech_mtime = os.path.getmtime(TECH_FILE)
tech_dt    = datetime.fromtimestamp(tech_mtime)
tech_age   = (_time_mod.time() - tech_mtime) / 3600
age_warn   = '⚠️ ' if tech_age > 72 else '✅'
print(f"  Data updated : {age_warn} "
      f"{tech_dt.strftime('%Y-%m-%d %H:%M')} "
      f"({tech_age:.0f}h ago)")
print("  Done.\n")

# ── GENERIC FILE HELPERS ──────────────────────────────────────
def load_port(filepath, cols=None):
    if not os.path.exists(filepath):
        return pd.DataFrame(columns=cols or [])
    try:
        df = pd.read_csv(filepath)
        if cols:
            for c in cols:
                if c not in df.columns:
                    df[c] = None
        return df
    except:
        return pd.DataFrame(columns=cols or [])

def save_port(df, filepath):
    df.to_csv(filepath, index=False)
    print(f"  ✅ Saved → {os.path.basename(filepath)}")

# ── STOCK LOOKUP HELPERS ──────────────────────────────────────
def get_current_price(symbol):
    rows = tech_df[tech_df['Symbol'] == symbol]
    if len(rows) > 0:
        return float(rows.iloc[0].get('Current Price', 0) or 0)
    return 0.0

def get_current_ml(symbol):
    rows = tech_df[tech_df['Symbol'] == symbol]
    if len(rows) > 0:
        r = rows.iloc[0]
        return (str(r.get('ML_Prediction', '—')),
                float(r.get('ML_Confidence', 0) or 0))
    return ('—', 0.0)

def get_current_trend(symbol):
    rows = tech_df[tech_df['Symbol'] == symbol]
    if len(rows) > 0:
        return str(rows.iloc[0].get('Sector Trend', '—'))
    return '—'

def compute_ema50_pct(symbol):
    rows = tech_df[tech_df['Symbol'] == symbol]
    if len(rows) == 0:
        return 0.0
    r     = rows.iloc[0]
    price = float(r.get('Current Price', 0) or 0)
    ema50 = float(r.get('EMA50', 0) or 0)
    if ema50 <= 0:
        return 0.0
    return round((price - ema50) / ema50 * 100, 1)

def validate_symbol(sym_input):
    sym     = sym_input.strip().upper()
    matches = tech_df[tech_df['Symbol'].str.upper() == sym]
    if len(matches) > 0:
        return matches.iloc[0]['Symbol']
    partial = tech_df[
        tech_df['Symbol'].str.upper().str.startswith(sym)]
    if len(partial) == 1:
        return partial.iloc[0]['Symbol']
    if len(partial) > 1:
        print(f"\n  Multiple matches for '{sym}':")
        for i, s in enumerate(partial['Symbol'].tolist()[:8], 1):
            print(f"    {i}. {s}")
        try:
            c = int(input("  Choose (number): ").strip())
            return partial.iloc[c - 1]['Symbol']
        except:
            return None
    print(f"  ⚠️  '{sym}' not found in universe.")
    return None

def show_stock_info(symbol):
    rows = tech_df[tech_df['Symbol'] == symbol]
    if len(rows) == 0:
        return
    r = rows.iloc[0]
    print(f"  → {symbol}  |  {r.get('Sector','?')}"
          f"  |  Rs{float(r.get('Current Price',0)):.2f}"
          f"  |  ML:{r.get('ML_Prediction','')}"
          f"  |  Conf:{float(r.get('ML_Confidence',0) or 0):.0f}%"
          f"  |  Tech:{float(r.get('Tech Score',0) or 0):.0f}"
          f"  |  EMA50%:{compute_ema50_pct(symbol):+.1f}%"
          f"  |  {short_trend(r.get('Sector Trend',''))}")

# ── FLAG HELPERS ──────────────────────────────────────────────
def ml_flag(ml_pred, ml_at_entry='Bullish Continual'):
    if (str(ml_at_entry) == 'Bullish Continual'
            and ml_pred in EXIT_FLAG_LABELS):
        return '🔴 EXIT'
    return ''

def trend_flag(current_trend, trend_at_entry):
    try:
        i_now   = TREND_ORDER.index(str(current_trend))
        i_entry = TREND_ORDER.index(str(trend_at_entry))
        if i_now > i_entry:
            return '🟡 REDUCE'
    except:
        pass
    return ''

def ema50_monitor_flag(symbol):
    pct = compute_ema50_pct(symbol)
    if 0 < pct < MONITOR_EMA50_PCT:
        return f'🟡 MONITOR (EMA50%={pct:.1f}%)'
    return ''

def stop_hit_flag(current_price, stop_price):
    try:
        if stop_price > 0 and float(current_price) < float(stop_price):
            return '🔴 STOP REF HIT'
    except:
        pass
    return ''

def corp_action_flag(symbol, entry_price):
    curr = get_current_price(symbol)
    if entry_price > 0 and curr > 0:
        drop = (entry_price - curr) / entry_price * 100
        if drop > CORP_ACTION_DROP_PCT:
            return f'⚠️  CORP ACTION? (drop {drop:.1f}%)'
    return ''

def gains_lock_flag(ml_pred, entry_price, current_price):
    if (ml_pred in ['Bearish Continual', 'Bearish']
            and current_price > entry_price):
        gain = ((current_price - entry_price)
                / entry_price * 100)
        return f'🏆 LOCK GAINS (+{gain:.1f}%)'
    return ''

# ── DATA FRESHNESS CHECK ──────────────────────────────────────
def check_data_freshness():
    """
    Compare tech file age against last optimizer run.
    Returns: 'MISSING' | 'OLD' | 'NEVER_RUN' | 'FRESH' | 'STALE'
    """
    if not os.path.exists(TECH_FILE):
        print("  ❌ technical_report_full.csv not found.")
        print("     Run run_weekly.py first.")
        return 'MISSING'

    tech_mtime   = os.path.getmtime(TECH_FILE)
    tech_age_hrs = (_time_mod.time() - tech_mtime) / 3600
    tech_dt      = datetime.fromtimestamp(tech_mtime)

    # Last optimizer run — check state file or tester folders
    last_run = None
    if os.path.exists(STATE_FILE):
        last_run = os.path.getmtime(STATE_FILE)
    # Also check any tester meta files
    if os.path.exists(OPT_TESTER_DIR):
        for tname in os.listdir(OPT_TESTER_DIR):
            meta_f = os.path.join(
                OPT_TESTER_DIR, tname, 'meta.txt')
            if os.path.exists(meta_f):
                try:
                    with open(meta_f) as f:
                        for line in f:
                            if line.startswith('last_run_ts='):
                                ts = float(
                                    line.split('=')[1].strip())
                                if last_run is None or ts > last_run:
                                    last_run = ts
                except:
                    pass

    if tech_age_hrs > 72:
        print(f"  ⚠️  Data is {tech_age_hrs/24:.1f} days old "
              f"({tech_dt.strftime('%Y-%m-%d %H:%M')})")
        print("     Run run_weekly.py for fresh signals.")
        return 'OLD'

    if last_run is None:
        print(f"  ✅ Data: {tech_dt.strftime('%Y-%m-%d %H:%M')} "
              f"({tech_age_hrs:.1f}h ago) — first run")
        return 'NEVER_RUN'

    if tech_mtime > last_run:
        last_dt = datetime.fromtimestamp(last_run)
        print(f"  ✅ New data: "
              f"{tech_dt.strftime('%Y-%m-%d %H:%M')} | "
              f"Last run: "
              f"{last_dt.strftime('%Y-%m-%d %H:%M')}")
        return 'FRESH'
    else:
        last_dt = datetime.fromtimestamp(last_run)
        print(f"  ℹ️  Data: "
              f"{tech_dt.strftime('%Y-%m-%d %H:%M')} | "
              f"Already ran: "
              f"{last_dt.strftime('%Y-%m-%d %H:%M')}")
        return 'STALE'

def freshness_gate(force_label='run anyway'):
    status = check_data_freshness()
    if status == 'MISSING':
        return False
    if status in ('OLD', 'STALE'):
        ans = input(
            f"\n  {force_label}? (y/n): "
        ).strip().lower()
        return ans == 'y'
    return True

# ── NIFTY 50 BENCHMARK ────────────────────────────────────────
def fetch_nifty_return(from_date_str):
    try:
        from_dt    = pd.Timestamp(from_date_str)
        fetch_from = (from_dt - pd.Timedelta(days=5)
                      ).strftime('%Y-%m-%d')
        tomorrow   = (datetime.now() + pd.Timedelta(days=1)
                      ).strftime('%Y-%m-%d')
        df_n = yf.download(
            '^NSEI', start=fetch_from, end=tomorrow,
            interval='1d', progress=False, auto_adjust=True)
        if isinstance(df_n.columns, pd.MultiIndex):
            df_n.columns = [c[0] for c in df_n.columns]
        df_n = df_n[df_n.index >= from_dt]
        if len(df_n) < 2:
            return None
        start_px   = float(df_n.iloc[0]['Close'])
        current_px = float(df_n.iloc[-1]['Close'])
        pct        = (current_px - start_px) / start_px * 100
        return round(start_px, 2), round(current_px, 2), round(pct, 2)
    except Exception as e:
        print(f"  ⚠️  Nifty fetch failed: {e}")
        return None

def fetch_nifty_series(from_date_str):
    try:
        from_dt    = pd.Timestamp(from_date_str)
        fetch_from = (from_dt - pd.Timedelta(days=5)
                      ).strftime('%Y-%m-%d')
        tomorrow   = (datetime.now() + pd.Timedelta(days=1)
                      ).strftime('%Y-%m-%d')
        df_n = yf.download(
            '^NSEI', start=fetch_from, end=tomorrow,
            interval='1d', progress=False, auto_adjust=True)
        if isinstance(df_n.columns, pd.MultiIndex):
            df_n.columns = [c[0] for c in df_n.columns]
        df_n  = df_n[df_n.index >= from_dt].copy()
        if len(df_n) < 2:
            return None, None
        base  = float(df_n.iloc[0]['Close'])
        dates = df_n.index.tolist()
        vals  = [float(p) / base * 100 for p in df_n['Close']]
        return dates, vals
    except:
        return None, None

# ── PORTFOLIO CHART ───────────────────────────────────────────
def build_portfolio_series(df_port, from_date_str):
    from_dt    = pd.Timestamp(from_date_str)
    all_dates  = set()
    stock_data = {}

    for _, row in df_port.iterrows():
        sym = str(row['Symbol'])
        if sym not in indicator_data:
            continue
        df_s = indicator_data[sym].copy()
        df_s = df_s[df_s.index >= from_dt]
        if len(df_s) < 2:
            continue
        stock_data[sym] = df_s
        all_dates.update(df_s.index.tolist())

    if not all_dates:
        return None, None, {}

    dates = sorted(all_dates)
    invested_total = sum(
        float(r.get('Entry_Price', 0)) *
        int(r.get('Quantity', 0))
        for _, r in df_port.iterrows()
    )
    if invested_total <= 0:
        return None, None, {}

    port_vals = []
    for dt in dates:
        day_val = 0
        for _, row in df_port.iterrows():
            sym   = str(row['Symbol'])
            qty   = int(row.get('Quantity', 0))
            entry = float(row.get('Entry_Price', 0))
            if sym not in stock_data:
                day_val += entry * qty
                continue
            df_s  = stock_data[sym]
            avail = df_s[df_s.index <= dt]
            px    = float(avail.iloc[-1]['Close']) \
                    if len(avail) > 0 else entry
            day_val += px * qty
        port_vals.append(day_val)

    base   = port_vals[0] if port_vals[0] > 0 else invested_total
    normed = [v / base * 100 for v in port_vals]

    stock_series = {}
    for sym, df_s in stock_data.items():
        s_vals = [float(p) for p in df_s['Close']]
        s_base = s_vals[0] if s_vals[0] > 0 else 1
        stock_series[sym] = (
            df_s.index.tolist(),
            [v / s_base * 100 for v in s_vals]
        )

    return dates, normed, stock_series

def plot_portfolio_vs_nifty(port_name, df_port,
                             from_date_str,
                             save_path=None):
    print(f"\n  Building chart for '{port_name}'...")
    dates_port, vals_port, stock_series = \
        build_portfolio_series(df_port, from_date_str)
    dates_nifty, vals_nifty = \
        fetch_nifty_series(from_date_str)

    if dates_port is None or len(dates_port) < 2:
        print("  ⚠️  Not enough price history to plot.")
        return None

    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor('#fafafa')
    ax.set_facecolor('#fafafa')

    for sym, (s_dates, s_vals) in stock_series.items():
        ax.plot(s_dates, s_vals,
                color='#cccccc', linewidth=0.8,
                alpha=0.6, zorder=1)
        if s_dates:
            ax.annotate(sym,
                        xy=(s_dates[-1], s_vals[-1]),
                        fontsize=6.5, color='#aaaaaa',
                        xytext=(3, 0),
                        textcoords='offset points',
                        va='center')

    if dates_nifty and vals_nifty:
        ax.plot(dates_nifty, vals_nifty,
                color='#888888', linewidth=1.8,
                linestyle='--', label='Nifty 50', zorder=2)
        nifty_ret = vals_nifty[-1] - 100
        ax.annotate(f'Nifty {nifty_ret:+.1f}%',
                    xy=(dates_nifty[-1], vals_nifty[-1]),
                    fontsize=8, color='#555555',
                    xytext=(5, 0),
                    textcoords='offset points',
                    va='center')

    port_label = port_name.replace('_', ' ').title()
    ax.plot(dates_port, vals_port,
            color='#1D9E75', linewidth=2.5,
            label=port_label, zorder=3)
    port_ret = vals_port[-1] - 100
    ax.annotate(f'{port_label} {port_ret:+.1f}%',
                xy=(dates_port[-1], vals_port[-1]),
                fontsize=9, color='#1D9E75',
                fontweight='bold', xytext=(5, 0),
                textcoords='offset points', va='center')

    ax.axhline(100, color='#dddddd', linewidth=1, zorder=0)
    ax.xaxis.set_major_formatter(
        mdates.DateFormatter('%d %b'))
    ax.xaxis.set_major_locator(
        mdates.WeekdayLocator(byweekday=0))
    plt.xticks(rotation=30, ha='right', fontsize=8)
    plt.yticks(fontsize=8)
    ax.set_ylabel('Indexed (100 = entry)', fontsize=9)
    ax.set_title(
        f"{port_label}  vs  Nifty 50\n"
        f"From {from_date_str}  to  {today_str}",
        fontsize=11, pad=12)
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(axis='y', color='#eeeeee', linewidth=0.8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()

    if save_path is None:
        save_path = os.path.join(
            REPORT_DIR,
            f"{port_name}_vs_nifty_{today_file}.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  📊 Chart saved → {os.path.basename(save_path)}")
    return save_path

# ── MODEL PORTFOLIO MANAGEMENT ────────────────────────────────
def list_model_portfolios():
    if not os.path.exists(OPT_MODEL_DIR):
        return []
    return sorted([
        f.replace('.csv', '')
        for f in os.listdir(OPT_MODEL_DIR)
        if f.endswith('.csv')
    ])

def add_stock_to_port(df, cols, port_name=''):
    new_records = []
    print("  Enter stocks to add. Type 'done' when finished.\n")
    while True:
        sym_input = input("  Symbol (or 'done'): ").strip()
        if sym_input.lower() == 'done':
            break
        symbol = validate_symbol(sym_input)
        if symbol is None:
            continue
        show_stock_info(symbol)

        try:
            curr_px = get_current_price(symbol)
            px_raw  = input(
                f"  Entry price (Rs) "
                f"[Enter = current {curr_px:.2f}]: "
            ).strip()
            entry_px = float(px_raw) if px_raw else curr_px
            if entry_px <= 0:
                raise ValueError
        except:
            print("  Invalid price. Skipping.")
            continue

        atr_info = atr_map.get(symbol)
        if atr_info:
            trend    = get_current_trend(symbol)
            atr_mult = (ATR_MULT_STR_UP if 'Strong' in trend
                        else ATR_MULT_UPTREND)
            stop_ref = round(
                entry_px - atr_info['ATR'] * atr_mult, 2)
            print(f"  ATR={atr_info['ATR']:.2f}  "
                  f"Stop ref=Rs{stop_ref:.2f}  "
                  f"(reference only)")
        else:
            stop_ref = round(entry_px * 0.93, 2)

        try:
            qty = int(input("  Quantity (shares): ").strip())
            if qty <= 0:
                raise ValueError
        except:
            print("  Invalid quantity. Skipping.")
            continue

        entry_date = input(
            "  Entry date (YYYY-MM-DD) "
            "[Enter = today]: "
        ).strip() or today_str
        notes = input("  Notes (optional): ").strip()

        ml_pred, ml_conf = get_current_ml(symbol)
        sec_trend        = get_current_trend(symbol)
        ema50_pct        = compute_ema50_pct(symbol)
        rows_t = tech_df[tech_df['Symbol'] == symbol]
        sec_rnk = float(
            rows_t.iloc[0].get('Sector Score', 0) or 0) \
            if len(rows_t) > 0 else 0.0
        cap_rnk = float(
            rows_t.iloc[0].get('Cap Score', 0) or 0) \
            if len(rows_t) > 0 else 0.0
        cap_cat = str(
            rows_t.iloc[0].get('Cap Category', '')) \
            if len(rows_t) > 0 else ''
        invested = round(entry_px * qty, 2)

        record = {
            'Symbol'                  : symbol,
            'Entry_Price'             : entry_px,
            'Entry_Date'              : entry_date,
            'Quantity'                : qty,
            'Cap_Category'            : cap_cat,
            'Invested'                : invested,
            'Sector_Rank_At_Entry'    : sec_rnk,
            'Cap_Rank_At_Entry'       : cap_rnk,
            'ML_At_Entry'             : ml_pred,
            'Sector_Trend_At_Entry'   : sec_trend,
            'EMA50_Pct_At_Entry'      : ema50_pct,
            'Optimizer_Score_At_Entry': 0.0,
            'Health_At_Entry'         : '—',
            'Stop_Price_Ref'          : stop_ref,
            'Portfolio_Name'          : port_name,
            'Notes'                   : notes,
        }
        print(f"  ✅ {symbol}  {qty}×Rs{entry_px:.2f}"
              f"=Rs{invested:,.0f}  "
              f"StopRef:Rs{stop_ref:.2f}  "
              f"ML:{ml_pred}  "
              f"{short_trend(sec_trend)}")
        new_records.append(record)

    if new_records:
        new_df = pd.DataFrame(new_records)
        for c in cols:
            if c not in new_df.columns:
                new_df[c] = None
        df = pd.concat(
            [df, new_df[cols]], ignore_index=True)
        df = df.drop_duplicates(
            subset=['Symbol'], keep='last')
    return df

def remove_stocks_from_port(df):
    if len(df) == 0:
        print("  Portfolio is empty.")
        return df
    print("\n  Current holdings:")
    for i, row in df.iterrows():
        print(f"    {i+1:>2}. {str(row['Symbol']):<14}"
              f"  Rs{float(row.get('Entry_Price',0)):.2f}"
              f" × {int(row.get('Quantity',0))}"
              f"  = Rs{float(row.get('Invested',0)):,.0f}")
    raw = input(
        "\n  Row numbers to remove "
        "(comma-separated): ").strip()
    try:
        indices = [int(x.strip()) - 1
                   for x in raw.split(',')]
        df = df.drop(
            df.index[indices]).reset_index(drop=True)
        print(f"  ✅ Removed {len(indices)} stock(s).")
    except:
        print("  Invalid input. No changes made.")
    return df

def update_quantity_price(df):
    if len(df) == 0:
        print("  Portfolio is empty.")
        return df
    print("\n  Current holdings:")
    for i, row in df.iterrows():
        print(f"    {i+1:>2}. {str(row['Symbol']):<14}"
              f"  Rs{float(row.get('Entry_Price',0)):.2f}"
              f" × {int(row.get('Quantity',0))}")
    try:
        idx = int(
            input("\n  Row number to edit: ").strip()) - 1
        row = df.iloc[idx]
        sym = str(row['Symbol'])

        px_raw = input(
            f"  New entry price [Enter = keep "
            f"{float(row.get('Entry_Price',0)):.2f}]: "
        ).strip()
        if px_raw:
            df.at[df.index[idx], 'Entry_Price'] = \
                float(px_raw)

        qty_raw = input(
            f"  New quantity [Enter = keep "
            f"{int(row.get('Quantity',0))}]: "
        ).strip()
        if qty_raw:
            df.at[df.index[idx], 'Quantity'] = \
                int(qty_raw)

        ep  = float(df.at[df.index[idx], 'Entry_Price'])
        qty = int(df.at[df.index[idx], 'Quantity'])
        df.at[df.index[idx], 'Invested'] = \
            round(ep * qty, 2)
        atr_info = atr_map.get(sym)
        trend    = str(row.get(
            'Sector_Trend_At_Entry', ''))
        atr_mult = (ATR_MULT_STR_UP if 'Strong' in trend
                    else ATR_MULT_UPTREND)
        stop_ref = (round(ep - atr_info['ATR'] * atr_mult, 2)
                    if atr_info else round(ep * 0.93, 2))
        df.at[df.index[idx], 'Stop_Price_Ref'] = stop_ref
        print(f"  ✅ Updated. "
              f"Invested=Rs{ep*qty:,.0f}  "
              f"StopRef=Rs{stop_ref:.2f}")
    except Exception as e:
        print(f"  Error: {e}. No changes made.")
    return df

# ── OPTION 2 : CREATE / MANAGE MODEL PORTFOLIO ───────────────
def run_create_model_portfolio():
    print("\n" + "─"*55)
    print("  MODEL PORTFOLIO — CREATE / MANAGE")
    print("─"*55)

    names = list_model_portfolios()
    if names:
        print("\n  Existing portfolios:")
        for i, n in enumerate(names, 1):
            fp  = os.path.join(OPT_MODEL_DIR, f"{n}.csv")
            df_ = load_port(fp)
            print(f"    {i}. {n}  ({len(df_)} stocks)")
        print(f"    {len(names)+1}. Create new portfolio")
        try:
            c = int(input(
                f"\n  Choice (1-{len(names)+1}): "
            ).strip())
        except:
            c = len(names) + 1

        if 1 <= c <= len(names):
            port_name = names[c - 1]
            filepath  = os.path.join(
                OPT_MODEL_DIR, f"{port_name}.csv")
            df = load_port(filepath, OPT_MODEL_COLS)
            print(f"\n  Managing '{port_name}' "
                  f"({len(df)} stocks)")
            print("\n    1. Add stocks")
            print("    2. Remove stocks")
            print("    3. Edit price / quantity")
            print("    4. Delete portfolio")
            print("    5. Back")
            sub = input("  Choice: ").strip()

            if sub == '1':
                df = add_stock_to_port(
                    df, OPT_MODEL_COLS, port_name)
                save_port(df, filepath)
            elif sub == '2':
                df = remove_stocks_from_port(df)
                save_port(df, filepath)
            elif sub == '3':
                df = update_quantity_price(df)
                save_port(df, filepath)
            elif sub == '4':
                confirm = input(
                    f"  Delete '{port_name}'? "
                    f"(yes/no): "
                ).strip().lower()
                if confirm == 'yes':
                    os.remove(filepath)
                    meta = os.path.join(
                        OPT_MODEL_DIR,
                        f"{port_name}_meta.txt")
                    if os.path.exists(meta):
                        os.remove(meta)
                    print(f"  ✅ Deleted.")
            return

    # Create new
    port_name = input(
        "\n  New portfolio name "
        "(e.g. 'metals_focus'): "
    ).strip().lower().replace(' ', '_')
    if not port_name:
        print("  Invalid name. Cancelled.")
        return
    filepath = os.path.join(
        OPT_MODEL_DIR, f"{port_name}.csv")
    if os.path.exists(filepath):
        print(f"  '{port_name}' already exists.")
        return

    capital = ask_capital(
        label=f"model portfolio '{port_name}'",
        default=200_000)
    df = pd.DataFrame(columns=OPT_MODEL_COLS)
    df = add_stock_to_port(
        df, OPT_MODEL_COLS, port_name)

    if len(df) == 0:
        print("  No stocks added. Not saved.")
        return

    meta_path = os.path.join(
        OPT_MODEL_DIR, f"{port_name}_meta.txt")
    with open(meta_path, 'w') as mf:
        mf.write(f"capital={capital}\n")
        mf.write(f"created={today_str}\n")

    save_port(df, filepath)
    print(f"\n  ✅ '{port_name}' created — "
          f"{len(df)} stocks, "
          f"Capital Rs{capital:,.0f}")

# ── OPTION 3 : REVIEW MODEL PORTFOLIO ────────────────────────
def run_review_model_portfolios():
    print("\n" + "─"*55)
    print("  MODEL PORTFOLIO — REVIEW")
    print("─"*55)

    names = list_model_portfolios()
    if not names:
        print("  No portfolios. Create one first (Option 2).")
        return

    print("\n  Select portfolio:")
    for i, n in enumerate(names, 1):
        fp  = os.path.join(OPT_MODEL_DIR, f"{n}.csv")
        df_ = load_port(fp)
        print(f"    {i}. {n}  ({len(df_)} stocks)")
    print(f"    {len(names)+1}. All (comparison)")

    try:
        c = int(input(
            f"\n  Choice (1-{len(names)+1}): "
        ).strip())
    except:
        c = len(names) + 1

    to_review = ([names[c - 1]]
                 if 1 <= c <= len(names)
                 else names)

    lines   = []
    summary = []

    def p(line=''): lines.append(str(line))

    p("=" * 80)
    p("  MODEL PORTFOLIO REVIEW")
    p(f"  {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    p("=" * 80)

    for port_name in to_review:
        filepath = os.path.join(
            OPT_MODEL_DIR, f"{port_name}.csv")
        df       = load_port(filepath, OPT_MODEL_COLS)
        if len(df) == 0:
            p(f"\n  {port_name} — empty")
            continue

        capital   = 0
        meta_path = os.path.join(
            OPT_MODEL_DIR, f"{port_name}_meta.txt")
        if os.path.exists(meta_path):
            with open(meta_path) as mf:
                for line in mf:
                    if line.startswith('capital='):
                        try:
                            capital = float(
                                line.split('=')[1].strip())
                        except:
                            pass

        total_inv  = 0
        total_curr = 0
        stock_lines = []

        entry_dates = pd.to_datetime(
            df['Entry_Date'], errors='coerce').dropna()
        port_from = (
            entry_dates.min().strftime('%Y-%m-%d')
            if len(entry_dates) > 0 else today_str)

        nifty_info = fetch_nifty_return(port_from)

        for _, row in df.iterrows():
            sym      = str(row['Symbol'])
            entry_px = float(row.get('Entry_Price', 0) or 0)
            qty      = int(row.get('Quantity', 0) or 0)
            invested = entry_px * qty
            total_inv += invested

            curr_px  = get_current_price(sym)
            curr_px  = curr_px if curr_px > 0 else entry_px
            curr_val = curr_px * qty
            total_curr += curr_val
            pl_pct   = ((curr_val - invested) / invested * 100
                        if invested > 0 else 0)

            curr_ml, curr_conf = get_current_ml(sym)
            ml_entry    = str(row.get(
                'ML_At_Entry', 'Bullish Continual'))
            trend_entry = str(row.get(
                'Sector_Trend_At_Entry', ''))
            curr_trend  = get_current_trend(sym)
            stop_ref    = float(
                row.get('Stop_Price_Ref', 0) or 0)

            f1 = ml_flag(curr_ml, ml_entry)
            f2 = trend_flag(curr_trend, trend_entry)
            f3 = ema50_monitor_flag(sym)
            f4 = stop_hit_flag(curr_px, stop_ref)
            f5 = corp_action_flag(sym, entry_px)
            f6 = gains_lock_flag(
                curr_ml, entry_px, curr_px)

            flags = '  '.join(
                f for f in [f1,f2,f3,f4,f5,f6] if f)

            stock_lines.append(
                f"  {sym:<14} "
                f"Rs{entry_px:.2f}→Rs{curr_px:.2f}  "
                f"P&L:{pl_pct:+.1f}%  "
                f"ML:{curr_ml[:16]} {curr_conf:.0f}%  "
                f"{short_trend(curr_trend)}"
                + (f"\n    {flags}" if flags else ""))

        port_pl     = total_curr - total_inv
        port_pl_pct = (port_pl / total_inv * 100
                       if total_inv > 0 else 0)

        nifty_line = ''
        if nifty_info:
            n_s, n_c, n_p = nifty_info
            alpha = port_pl_pct - n_p
            nifty_line = (
                f"  Nifty 50: {n_p:+.1f}%  "
                f"({n_s:.0f}→{n_c:.0f})  "
                f"Alpha: {alpha:+.1f}%")

        summary.append({
            'name'      : port_name,
            'invested'  : total_inv,
            'curr_val'  : total_curr,
            'pl_abs'    : port_pl,
            'pl_pct'    : port_pl_pct,
            'stocks'    : len(df),
            'nifty_pct' : nifty_info[2] if nifty_info else None,
            'from_date' : port_from,
        })

        p(f"\n{'─'*80}")
        p(f"  {port_name.upper().replace('_',' ')}")
        p(f"  Stocks:{len(df)}  "
          f"Invested:Rs{total_inv:,.0f}  "
          f"Current:Rs{total_curr:,.0f}  "
          f"P&L:Rs{port_pl:+,.0f} ({port_pl_pct:+.1f}%)")
        if nifty_line:
            p(nifty_line)
        p('─'*80)
        for sl in stock_lines:
            p(sl)

    if len(summary) > 1:
        ss = sorted(summary,
                    key=lambda x: x['pl_pct'],
                    reverse=True)
        p(f"\n{'='*80}")
        p("  RANKING  (by P&L %)")
        p('='*80)
        for i, s in enumerate(ss, 1):
            medal = {1:'🥇',2:'🥈',3:'🥉'}.get(
                i, f'#{i}')
            n_s = (f"{s['nifty_pct']:>+6.1f}%"
                   if s['nifty_pct'] is not None
                   else '     —')
            a_s = (f"{s['pl_pct']-s['nifty_pct']:>+6.1f}%"
                   if s['nifty_pct'] is not None
                   else '     —')
            p(f"  {medal}  "
              f"{s['name'].replace('_',' '):<28} "
              f"{s['stocks']:>4}stk  "
              f"Rs{s['pl_abs']:>+10,.0f}  "
              f"{s['pl_pct']:>+6.1f}%  "
              f"Nifty:{n_s}  Alpha:{a_s}")
        p('='*80)

    for line in lines:
        print(line)

    report_path = os.path.join(
        REPORT_DIR, f'model_review_{today_file}.txt')
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"\n  Report saved → "
          f"{os.path.basename(report_path)}")

    if input("\n  Generate chart(s) vs Nifty? (y/n): "
             ).strip().lower() == 'y':
        for port_name in to_review:
            fp = os.path.join(
                OPT_MODEL_DIR, f"{port_name}.csv")
            df = load_port(fp, OPT_MODEL_COLS)
            if len(df) == 0:
                continue
            ed = pd.to_datetime(
                df['Entry_Date'],
                errors='coerce').dropna()
            pf = (ed.min().strftime('%Y-%m-%d')
                  if len(ed) > 0 else today_str)
            plot_portfolio_vs_nifty(port_name, df, pf)

# ── MAIN MENU LOOP ────────────────────────────────────────────
def run_optimizer_menu():
    while True:
        print(MENU)
        choice = input("  Enter choice (1-7): ").strip()

        if choice == '1':
            print("\n" + "─"*55)
            print("  FRESH RECOMMENDATIONS")
            print("─"*55)
            if not freshness_gate('Run anyway'):
                input("  Press Enter to return...")
                continue
            # run_pipeline defined in Cell 3
            _df_all, _df_final, _status = run_pipeline(
                label='fresh recommendations')
            if _status not in ('NO_STOCKS_L1', 'NO_BUY'):
                print_pipeline_table(
                    _df_final,
                    title='FRESH RECOMMENDATIONS')
            input("\n  Press Enter to return to menu...")

        elif choice == '2':
            run_create_model_portfolio()

        elif choice == '3':
            run_review_model_portfolios()

        elif choice == '4':
            # Auto-tester — Cell 4
            run_auto_tester()

        elif choice == '5':
            run_build_actual_portfolio()

        elif choice == '6':
            run_review_actual_portfolio()

        elif choice == '7':
            print("\n  Exiting optimizer.")
            break

        else:
            print("  Invalid choice. Enter 1–7.")

# ── COMPLETION ────────────────────────────────────────────────
print("✅  Cell 2 complete — all helpers loaded")
print("     Run Cell 3 next, then Cell 4,")
print("     then call run_optimizer_menu()")

Loading data...
  Tech data    : 752 stocks
  ATR data     : 752 stocks
  State file   : 23 rows — comparing vs last run
  Data updated : ✅ 2026-04-30 18:08 (18h ago)
  Done.

✅  Cell 2 complete — all helpers loaded
     Run Cell 3 next, then Cell 4,
     then call run_optimizer_menu()


In [14]:
# ============================================================
# DAY 15 — Portfolio Optimizer
# Cell 3 : Core engine — 4-layer pipeline
#
# Layer 1 — Filter    : 5 hard gates + Tier 3 block
# Layer 2 — Score     : composite 0-140, three multipliers
#                       sector × tier × S6(20d range)
# Layer 3 — Size      : equal allocation, dynamic sector cap
# Layer 4 — Health    : 7 signals, score -6 to +6
#
# All changes from design sessions:
#   Gate 6     : blocks Tier 3 only (was blocking Tier 2)
#   EMA50%     : bell curve scoring, peaks 8-15%
#   Score      : Tier mult + S6 mult added
#   Health S4  : RSI overbought reversal penalty added
#                (RSI falling FROM overbought + price falling)
#   Health S6  : 20d range position (upside room)
#   Health S7  : MACD+Price slope (double penalty + recovery)
#                + GALLANTT fix (hist already negative check)
#   Pipeline   : shows stock list BEFORE asking capital/count
#                MIXED and NEUTRAL = valid entries
# ============================================================

# ── LAYER 1 : FILTER ─────────────────────────────────────────
def filter_candidates(df, verbose=True):
    """
    5 hard gates + Tier 3 block.

    Gate 6 change: blocks Tier 3 only (not Tier 2).
    Tier 1 vs Tier 2 preference handled in Layer 2 scoring.
    Result: ~23-53 stocks passing depending on market conditions.

    Confirmed by 15-day test:
      FORCEMOT  EMA50%=+1.8% → gate 3 → avoided -11.8%
      ORICONENT EMA50%=+0.7% → gate 3 → avoided -4.8%
      ZEAL      Sideways     → gate 5 → avoided -4.2%
    """
    total   = len(df)
    rejects = {}
    working = df.copy()

    # Gate 1 — ML label
    mask = working['ML_Prediction'].astype(str) \
           == 'Bullish Continual'
    rejects['ML not Bull Cont'] = (~mask).sum()
    working = working[mask].copy()

    # Gate 2 — ML confidence
    working['ML_Confidence_f'] = pd.to_numeric(
        working['ML_Confidence'], errors='coerce').fillna(0)
    mask = working['ML_Confidence_f'] >= MIN_ML_CONF
    rejects[f'ML conf < {MIN_ML_CONF}%'] = (~mask).sum()
    working = working[mask].copy()

    # Gate 3 — EMA50%
    def _ema50_pct(row):
        try:
            price = float(row.get('Current Price', 0) or 0)
            ema50 = float(row.get('EMA50', 0) or 0)
            if ema50 <= 0:
                return 0.0
            return (price - ema50) / ema50 * 100
        except:
            return 0.0

    working['EMA50_Pct'] = working.apply(_ema50_pct, axis=1)
    mask = working['EMA50_Pct'] >= MIN_EMA50_PCT
    rejects[f'EMA50% < {MIN_EMA50_PCT}%'] = (~mask).sum()
    working = working[mask].copy()

    # Gate 4 — Tech Score
    working['Tech Score_f'] = pd.to_numeric(
        working['Tech Score'], errors='coerce').fillna(0)
    mask = working['Tech Score_f'] >= MIN_TECH_SCORE
    rejects[f'Tech < {MIN_TECH_SCORE}'] = (~mask).sum()
    working = working[mask].copy()

    # Gate 5 — Sector Trend
    # Exact strings from run_weekly.py derive_trend_label()
    mask = working['Sector Trend'].astype(str).isin(
        VALID_SECTOR_TRENDS)
    rejects['Sector not uptrend'] = (~mask).sum()
    working = working[mask].copy()

    # Gate 6 — Tier 3 blocked only
    # Tier 1 and Tier 2 both qualify here
    # Tier preference expressed via score multiplier in Layer 2
    mask = ~working['Tier'].astype(str).str.startswith(
        'TIER 3')
    rejects['Tier 3'] = (~mask).sum()
    working = working[mask].copy()

    passed = len(working)

    if verbose:
        print(f"  Filter results : {total} → {passed} stocks")
        print(f"  {'Gate':<30} {'Rejected':>8}")
        print("  " + "─" * 40)
        for reason, count in rejects.items():
            if count > 0:
                print(f"  {reason:<30} {count:>8}")
        print(f"  {'─'*40}")
        print(f"  {'Passed':<30} {passed:>8}")
        if passed == 0:
            print(f"\n  ⚠️  Zero stocks passed all filters.")
            print(f"     Market conditions do not meet criteria.")
            print(f"     Holding cash. No action taken.")
        elif passed < MIN_STOCKS:
            print(f"\n  ⚠️  Only {passed} stocks — "
                  f"low diversification.")

    return working.reset_index(drop=True)


# ── LAYER 2 : SCORE ───────────────────────────────────────────
def _ema50_component(ema50_pct):
    """
    EMA50% → 0-100 component score.
    Bell curve — peaks at 8-15% (close to EMA, confirmed uptrend).
    Reduces beyond 20% (extended, less runway left).

    Design rationale:
      Stock at +10% above EMA50 in rising trend has MORE upside
      than stock at +30% above EMA50. The +10% stock is early
      in its move. The +30% stock has already run far.

      Had we scored MTARTECH when it was at +9% (before the big
      run), it would have scored 99 — top of the list.
      At +33% it scores 40 — still included, lower priority.

      This correctly ranks NEAR-EMA rising stocks above
      EXTENDED stocks — more runway = higher priority.

    Bands:
      <5%   : 0   — below filter threshold
      5-8%  : 50  — entry zone
      8-15% : 99  — SWEET SPOT: close to EMA, confirmed move
      15-20%: 85  — moderate extension, still good
      20-25%: 70  — getting extended
      25-30%: 55  — significantly extended
      >30%  : 40  — over-extended, lower priority (not excluded)
    """
    e = float(ema50_pct)
    if e < 5:     return 0
    elif e < 8:   return 50
    elif e <= 15: return 99
    elif e <= 20: return 85
    elif e <= 25: return 70
    elif e <= 30: return 55
    else:         return 40


def compute_optimizer_score(row):
    """
    Composite score 0-140.

    raw   = W_EMA50*ema + W_TECH*tech + W_MLCONF*ml
              + W_RANKS*ranks + W_F25D*f25d
    final = raw x sector_mult x tier_mult x s6_mult

    Three multipliers applied after raw score:
      sector_mult : Str Up=1.3x, Uptrend=1.0x
      tier_mult   : Tier1=1.0x, Tier2=0.9x
      s6_mult     : 20d range 40-75% → S6_SCORE_MULT(1.08x)
                    outside sweet spot → 1.0x

    S6 multiplier: stock at 40-75% of 20d range has bounced
    from monthly lows and has room before monthly resistance.
    8% bonus surfaces these stocks over extended ones.
    Confirmed: SARDAEN(R20=73%) scores higher than GALLANTT(R20=79%)
    """
    try:
        ema50_pct = float(row.get('EMA50_Pct', 0) or 0)
        tech      = float(row.get('Tech Score_f',
                          row.get('Tech Score', 0)) or 0)
        ml_conf   = float(row.get('ML_Confidence_f',
                          row.get('ML_Confidence', 0)) or 0)
        sec_rnk   = float(row.get('Sector Score', 0) or 0)
        cap_rnk   = float(row.get('Cap Score',    0) or 0)
        f25d      = float(row.get('Forecast_25d_Pct', 0) or 0)
        trend     = str(row.get('Sector Trend', ''))
        tier      = str(row.get('Tier', ''))
        symbol    = str(row.get('Symbol', ''))

        # Component scores (all 0-100)
        c_ema50 = _ema50_component(ema50_pct)
        c_tech  = min(100, max(0, tech))
        c_ml    = min(100, max(0, (ml_conf - 40) / 45 * 100))
        c_ranks = min(100, (sec_rnk + cap_rnk) / 2 * 10)
        c_f25d  = min(100, max(0, f25d / 15 * 100))

        raw = (W_EMA50  * c_ema50 +
               W_TECH   * c_tech  +
               W_MLCONF * c_ml    +
               W_RANKS  * c_ranks +
               W_F25D   * c_f25d)

        # Multiplier 1: sector trend
        sector_mult = SECTOR_TREND_MULT.get(trend, 0.0)

        # Multiplier 2: tier
        tier_mult = (TIER1_MULT
                     if tier.startswith('TIER 1')
                     else TIER2_MULT)

        # Multiplier 3: S6 — 20d range sweet spot
        # Computed here so ranking reflects upside room
        # Layer 4 compute_health also computes independently
        # for the health label — not double counting
        s6_mult = 1.0
        if symbol in indicator_data:
            try:
                df_s = indicator_data[symbol]
                if len(df_s) >= 20:
                    last20   = df_s.iloc[-20:]
                    high_20d = float(last20['High'].max())
                    low_20d  = float(last20['Low'].min())
                    close    = float(df_s.iloc[-1]['Close'])
                    rng_20d  = high_20d - low_20d
                    if rng_20d > 0:
                        pos20 = (close - low_20d) / rng_20d
                        if 0.40 <= pos20 <= 0.75:
                            s6_mult = S6_SCORE_MULT  # 1.08
            except:
                pass

        return round(raw * sector_mult * tier_mult * s6_mult, 1)

    except:
        return 0.0


# ── LAYER 3 : POSITION SIZING ─────────────────────────────────
def _dynamic_sector_cap(df_filtered):
    """
    Sector cap scales with qualifying sectors.
    min(MAX_SECTOR_ABSOLUTE, 1/n_sectors x 1.5)

    Bear market (1-2 sectors): higher cap → concentrate strength
    Bull market (6+ sectors) : lower cap → natural diversification
    """
    n = max(1, df_filtered['Sector'].nunique()
            if 'Sector' in df_filtered.columns else 1)
    return round(
        min(MAX_SECTOR_ABSOLUTE, (1.0 / n) * 1.5), 4)


def compute_position_sizes(df_scored, capital, n_stocks):
    """
    Equal allocation: capital / n_stocks per stock.
    Confirmed by 15-day test: 0.28% difference vs weighted.
    Stock selection matters 30x more than allocation method.

    Three caps:
      Cap A: per stock  <= MAX_STOCK_PCT (15%)
      Cap B: per sector <= dynamic cap   (max 60%)
      Cap C: total      <= capital        (100%)

    ATR stop stored as reference only.
    Exit by indicators (ML/Layer4), not stop breach.
    """
    if len(df_scored) == 0:
        return df_scored.copy()

    df = df_scored.head(n_stocks).copy().sort_values(
        'Optimizer_Score', ascending=False
    ).reset_index(drop=True)

    dyn_sector_cap = _dynamic_sector_cap(df)
    max_per_stock  = capital * MAX_STOCK_PCT
    max_per_sector = capital * dyn_sector_cap
    alloc_base     = capital / n_stocks

    sector_deployed = {}
    total_deployed  = 0.0
    results         = []

    for _, row in df.iterrows():
        sym    = str(row['Symbol'])
        price  = float(row.get('Current Price', 0) or 0)
        trend  = str(row.get('Sector Trend', ''))
        sector = str(row.get('Sector', 'Unknown'))

        # ATR reference stop
        atr_info = atr_map.get(sym)
        if atr_info:
            atr     = atr_info['ATR']
            atr_pct = atr_info['ATR_pct']
        else:
            atr     = price * 0.02 if price > 0 else 0
            atr_pct = 2.0
        atr_mult       = (ATR_MULT_STR_UP
                          if 'Strong' in trend
                          else ATR_MULT_UPTREND)
        stop_price_ref = round(price - atr * atr_mult, 2) \
                         if price > 0 else 0

        # Cap C — capital fully deployed
        if total_deployed >= capital:
            row_out = row.copy()
            row_out['Allocation']       = 0
            row_out['Shares']           = 0
            row_out['Invested']         = 0
            row_out['Invested_Pct']     = 0
            row_out['Sector_Total_Pct'] = 0
            row_out['ATR']              = round(atr, 2)
            row_out['ATR_Pct']          = round(atr_pct, 2)
            row_out['Stop_Price_Ref']   = stop_price_ref
            row_out['Size_Note']        = 'Full'
            results.append(row_out)
            continue

        if price <= 0:
            continue

        alloc = alloc_base
        note  = ''

        # Cap A — per stock
        if alloc > max_per_stock:
            alloc = max_per_stock
            note  = 'S'

        # Cap B — per sector
        sect_used = sector_deployed.get(sector, 0)
        if sect_used >= max_per_sector:
            row_out = row.copy()
            row_out['Allocation']       = 0
            row_out['Shares']           = 0
            row_out['Invested']         = 0
            row_out['Invested_Pct']     = 0
            row_out['Sector_Total_Pct'] = round(
                sect_used / capital * 100, 1)
            row_out['ATR']              = round(atr, 2)
            row_out['ATR_Pct']          = round(atr_pct, 2)
            row_out['Stop_Price_Ref']   = stop_price_ref
            row_out['Size_Note']        = 'Sec'
            results.append(row_out)
            continue

        room_sector = max_per_sector - sect_used
        if alloc > room_sector:
            alloc = room_sector
            note  = (note + '+Sec') if note else 'Sec'

        room_total = capital - total_deployed
        if alloc > room_total:
            alloc = room_total
            note  = (note + '+Tot') if note else 'Tot'

        shares   = max(1, int(alloc / price))
        invested = round(shares * price, 2)
        invested = min(invested, max_per_stock,
                       room_sector, room_total)
        shares   = max(1, int(invested / price))
        invested = round(shares * price, 2)

        sector_deployed[sector] = (
            sector_deployed.get(sector, 0) + invested)
        total_deployed += invested

        row_out = row.copy()
        row_out['Allocation']       = round(alloc, 2)
        row_out['Shares']           = shares
        row_out['Invested']         = invested
        row_out['Invested_Pct']     = round(
            invested / capital * 100, 1)
        row_out['Sector_Total_Pct'] = round(
            sector_deployed[sector] / capital * 100, 1)
        row_out['ATR']              = round(atr, 2)
        row_out['ATR_Pct']          = round(atr_pct, 2)
        row_out['Stop_Price_Ref']   = stop_price_ref
        row_out['Size_Note']        = note
        results.append(row_out)

    return pd.DataFrame(results)


# ── LAYER 4 : MOMENTUM HEALTH ─────────────────────────────────
def compute_health(symbol):
    """
    7-signal momentum health using last 20 candles.
    All arithmetic — no candlestick subjectivity.
    Score range: -6 to +6

    S1 — 5-day price momentum
         close_now vs close_5d_ago
         >0 → +1   <0 → -1

    S2 — 5-day range position
         (close - 5d_low) / (5d_high - 5d_low)
         >=60% → +1   <=40% → -1   middle → 0
         Catches peaked stocks (Torrent Power = 21% → -1)
         Rewards bouncing stocks (closing near highs)

    S3 — EMA20 slope over 5 days
         ema20_now > ema20_5d_ago → +1   else -1
         Note: EMA20 lags price — stays positive during
         brief pullbacks (correctly so — trend intact)

    S4 — RSI direction + overbought reversal check
         Rising AND rsi<75           → +1 (momentum building)
         Falling FROM overbought:
           rsi>75 AND falling AND price falling → -2 (OB reversal)
           rsi>75 AND falling (price not yet falling) → -1 (warning)
         Falling AND rsi>65          → -1 (fading at highs)

         Key: RSI being overbought alone is NOT a penalty.
         Only penalised when RSI is FALLING FROM overbought
         (with or without price confirmation).
         A stock can stay overbought for weeks in a strong trend.

         RAMRAT at RSI=77.6: no penalty today (RSI not falling yet)
         RAMRAT next week if RSI falls to 72 with price falling:
           → fires -2 → score drops → RETREATING

    S5 — Higher low + higher close (recovery bonus)
         low_now > low_prev5 AND close_now > close_5d_ago
         → +1 (uptrend resuming after retracement)

    S6 — 20-day range position (upside room signal)
         (close - 20d_low) / (20d_high - 20d_low)
         40-75% → +1 SWEET SPOT: bounced from lows, room to run
                     ABSLAMC pattern = fresh bounce off EMA50
         0-39%  →  0 still in pullback, not ready
         76-100%→  0 extended, already near highs (no bonus)

    S7 — MACD Hist slope + price direction
         Case A: hist falling >1 point AND price falling
           → confirmed momentum loss → -2 DOUBLE PENALTY
           NAVA: Hist 7.9→-0.1 slope=-3.62 + price falling → -2

         Case B-ext: hist already negative AND falling from 5d
           AND price falling
           → catches GALLANTT where hist was already negative
           → -1 (single penalty)

         Case C: hist rising >0.5 (turning up after negative)
           → momentum resuming → +1 RECOVERY SIGNAL

         Case D: divergence or flat → 0 neutral

    Labels (-6 to +6):
      +4/+6  → RISING     clean entry, fresh momentum
      +1/+3  → MIXED      valid entry, some signals diverging
       0     → NEUTRAL    consolidating, watch
      -1/-6  → RETREATING avoid
    """
    result = {
        'Health_Score'  : 0,
        'Health_Label'  : 'UNKNOWN',
        'S1_Momentum'   : 0,
        'S2_Range5d'    : 0,
        'S3_EMA20'      : 0,
        'S4_RSI'        : 0,
        'S5_Recovery'   : 0,
        'S6_Range20d'   : 0,
        'S7_MACD'       : 0,
        'Health_Detail' : '—',
    }

    if symbol not in indicator_data:
        result['Health_Label']  = 'NO_DATA'
        result['Health_Detail'] = 'Not in indicator_data'
        return result

    try:
        df_s = indicator_data[symbol].copy()

        if len(df_s) < 21:
            result['Health_Label']  = 'INSUFFICIENT'
            result['Health_Detail'] = f'Only {len(df_s)} candles'
            return result

        # Use last 20 candles
        last20 = df_s.iloc[-20:].copy()
        last5  = df_s.iloc[-5:].copy()
        prev5  = df_s.iloc[-10:-5].copy()

        close_now    = float(last20.iloc[-1]['Close'])
        close_5d_ago = float(last20.iloc[-5]['Close'])
        close_3d_ago = float(last20.iloc[-3]['Close'])

        high_5d   = float(last5['High'].max())
        low_5d    = float(last5['Low'].min())
        high_20d  = float(last20['High'].max())
        low_20d   = float(last20['Low'].min())
        low_prev5 = float(prev5['Low'].min())

        rsi_now    = float(
            last20.iloc[-1].get('RSI', 50) or 50)
        rsi_3d_ago = float(
            last20.iloc[-3].get('RSI', 50) or 50)

        ema20_now    = float(
            last20.iloc[-1].get('EMA20', 0) or 0)
        ema20_5d_ago = float(
            last20.iloc[-5].get('EMA20', 0) or 0)

        # price_slope used in both S4 and S7
        price_slope = close_now - close_3d_ago

        score   = 0
        details = []

        # S1 — 5-day price momentum
        if close_5d_ago > 0:
            mom = ((close_now - close_5d_ago)
                   / close_5d_ago * 100)
            if mom > 0:
                result['S1_Momentum'] = 1
                score += 1
                details.append(f'Mom:+{mom:.1f}%↑')
            else:
                result['S1_Momentum'] = -1
                score -= 1
                details.append(f'Mom:{mom:.1f}%↓')

        # S2 — 5-day range position
        range_5d = high_5d - low_5d
        if range_5d > 0:
            pos5 = (close_now - low_5d) / range_5d
            if pos5 >= 0.6:
                result['S2_Range5d'] = 1
                score += 1
                details.append(f'R5:{pos5:.0%}↑')
            elif pos5 <= 0.4:
                result['S2_Range5d'] = -1
                score -= 1
                details.append(f'R5:{pos5:.0%}↓')
            else:
                details.append(f'R5:{pos5:.0%}→')

        # S3 — EMA20 slope
        if ema20_5d_ago > 0:
            slope = ((ema20_now - ema20_5d_ago)
                     / ema20_5d_ago * 100)
            if slope > 0:
                result['S3_EMA20'] = 1
                score += 1
                details.append(f'E20:+{slope:.2f}%↑')
            else:
                result['S3_EMA20'] = -1
                score -= 1
                details.append(f'E20:{slope:.2f}%↓')

        # S4 — RSI direction + overbought reversal
        # RSI being overbought alone is NOT a penalty.
        # Only penalised when RSI is FALLING FROM overbought.
        rsi_chg = rsi_now - rsi_3d_ago

        if rsi_now > 75 and rsi_chg < -2 and price_slope < 0:
            # RSI falling sharply FROM overbought AND price falling
            # = overbought reversal confirmed — strongest signal
            # RAMRAT next week if RSI 77→70 with price down: fires -2
            result['S4_RSI'] = -2
            score -= 2
            details.append(
                f'RSI:OB↓{rsi_now:.0f}+P↓')

        elif rsi_now > 75 and rsi_chg < 0:
            # RSI falling from overbought (price not yet falling)
            # Early warning — single penalty
            result['S4_RSI'] = -1
            score -= 1
            details.append(
                f'RSI:OB↓{rsi_now:.0f}')

        elif rsi_chg > 0 and rsi_now < 75:
            # Rising and not overbought — healthy momentum
            result['S4_RSI'] = 1
            score += 1
            details.append(
                f'RSI:{rsi_3d_ago:.0f}→{rsi_now:.0f}↑')

        elif rsi_chg < 0 and rsi_now > 65:
            # Falling from high zone — momentum fading
            result['S4_RSI'] = -1
            score -= 1
            details.append(
                f'RSI:{rsi_3d_ago:.0f}→{rsi_now:.0f}↓')

        else:
            details.append(f'RSI:{rsi_now:.0f}→')

        # S5 — Recovery bonus
        if (low_5d > low_prev5 and
                close_now > close_5d_ago):
            result['S5_Recovery'] = 1
            score += 1
            details.append('Rec✓')

        # S6 — 20-day range position (upside room)
        range_20d = high_20d - low_20d
        if range_20d > 0:
            pos20 = (close_now - low_20d) / range_20d
            if 0.40 <= pos20 <= 0.75:
                result['S6_Range20d'] = 1
                score += 1
                details.append(f'R20:{pos20:.0%}✓')
            elif pos20 > 0.75:
                details.append(f'R20:{pos20:.0%}→')
            else:
                details.append(f'R20:{pos20:.0%}↓')

        # S7 — MACD Hist slope + price direction
        # Uses 'MACD Hist' column from indicator_data
        # run_weekly.py stores as 'MACD Hist'
        try:
            # Find MACD Hist column
            macd_hist_col = None
            for col in df_s.columns:
                if col in ('MACD Hist', 'MACD_Hist',
                           'macd_hist'):
                    macd_hist_col = col
                    break

            if macd_hist_col:
                hist_now = float(
                    last20.iloc[-1].get(
                        macd_hist_col, 0) or 0)
                hist_3d  = float(
                    last20.iloc[-3].get(
                        macd_hist_col, 0) or 0)
                hist_5d  = float(
                    last20.iloc[-5].get(
                        macd_hist_col, 0) or 0)

                hist_slope = hist_now - hist_3d

                if hist_slope < -1.0 and price_slope < 0:
                    # Case A: MACD and price both falling sharply
                    # Double penalty — confirmed momentum loss
                    # NAVA: hist 3.51→-0.11 slope=-3.62
                    #       price falling → fires -2
                    result['S7_MACD'] = -2
                    score -= 2
                    details.append(
                        f'MACD↓P↓({hist_slope:.1f})')

                elif (hist_now < -1.0 and
                      hist_5d > hist_now and
                      price_slope < 0):
                    # Case B-ext: MACD hist already negative AND
                    # still falling from 5 days ago AND price falling
                    # Catches GALLANTT where hist was already negative
                    # before the 3-day lookback window
                    result['S7_MACD'] = -1
                    score -= 1
                    details.append(
                        f'MACD-ve({hist_now:.1f})')

                elif hist_slope > 0.5 and hist_now > hist_3d:
                    # Case C: MACD Hist turning up (recovery signal)
                    # Momentum resuming after being negative
                    # NAVA next week: -0.1→+1.5→+3.0 → fires +1
                    result['S7_MACD'] = 1
                    score += 1
                    details.append(
                        f'MACD↑({hist_slope:+.1f})')

                else:
                    # Case D: divergence or flat
                    result['S7_MACD'] = 0
                    details.append('MACD→')

            else:
                details.append('MACD:N/A')

        except:
            result['S7_MACD'] = 0
            details.append('MACD:err')

        # Label assignment
        # Score range -6 to +6
        result['Health_Score'] = score
        result['Health_Label'] = (
            'RISING'     if score >= 4 else
            'MIXED'      if score >= 1 else
            'NEUTRAL'    if score == 0 else
            'RETREATING'
        )
        result['Health_Detail'] = '  '.join(details)

    except Exception as e:
        result['Health_Label']  = 'ERROR'
        result['Health_Detail'] = str(e)[:50]

    return result


def apply_health_check(df_sized):
    """Run Layer 4 on all stocks. Adds health columns."""
    if len(df_sized) == 0:
        return df_sized

    health_rows = [
        compute_health(str(row['Symbol']))
        for _, row in df_sized.iterrows()
    ]
    df_health = pd.DataFrame(health_rows)
    df_out    = df_sized.reset_index(drop=True)
    for col in df_health.columns:
        df_out[col] = df_health[col].values
    return df_out


# ── COMBINED PIPELINE ─────────────────────────────────────────
def run_pipeline(label='candidates',
                 exclude_symbols=None,
                 verbose=True,
                 capital=None,
                 n_stocks=None):
    """
    Full 4-layer pipeline.

    NEW FLOW — shows stock list BEFORE asking capital/count:
      Phase A: Layer 1 + 2 + 4 (no capital needed)
               Shows all qualifying stocks with health labels
      Phase B: ask count → ask capital → Layer 3 sizing

    Args:
      label           : display label
      exclude_symbols : already-held symbols to skip
      verbose         : print progress and ask inputs
      capital         : pre-set capital (skips prompt)
      n_stocks        : pre-set count (skips prompt)

    Returns:
      df_all    : all Layer 1 survivors, scored + health
      df_final  : sized stocks
      status    : 'OK'|'NO_STOCKS_L1'|'NO_BUY'|
                  'NO_STOCKS_L4'|'WARN_FEW'
      capital   : capital used
      n_stocks  : count used
    """
    if verbose:
        print(f"\n  Running pipeline [{label}]...\n")

    # Phase A

    # Layer 1
    df_f = filter_candidates(tech_df, verbose=verbose)

    if len(df_f) == 0:
        return (pd.DataFrame(), pd.DataFrame(),
                'NO_STOCKS_L1', 0, 0)

    # Exclude already-held symbols
    if exclude_symbols and len(exclude_symbols) > 0:
        before = len(df_f)
        df_f   = df_f[~df_f['Symbol'].isin(
            list(exclude_symbols))]
        excl   = before - len(df_f)
        if excl > 0 and verbose:
            print(f"  Excluded {excl} already-held stock(s)")

    if len(df_f) == 0:
        if verbose:
            print("  ⚠️  All qualifying stocks already held.")
        return (df_f, pd.DataFrame(),
                'NO_BUY', 0, 0)

    # Layer 2 — score all qualifying stocks
    df_f['Optimizer_Score'] = df_f.apply(
        compute_optimizer_score, axis=1)
    df_f = df_f.sort_values(
        'Optimizer_Score', ascending=False
    ).reset_index(drop=True)

    # Layer 4 — health check ALL qualifying stocks
    # Applied BEFORE any count cap
    if verbose:
        print(f"\n  Running Layer 4 health on "
              f"all {len(df_f)} qualifying stocks...")
    df_f = apply_health_check(df_f)

    # Show full candidate list BEFORE asking capital
    if verbose:
        _show_candidate_list(df_f)

    # Count health labels
    rising     = df_f[df_f['Health_Label'] == 'RISING']
    mixed      = df_f[df_f['Health_Label'] == 'MIXED']
    neutral    = df_f[df_f['Health_Label'] == 'NEUTRAL']
    retreating = df_f[df_f['Health_Label'] == 'RETREATING']

    # RISING + MIXED + NEUTRAL all valid entries
    investable = len(rising) + len(mixed) + len(neutral)

    if investable == 0:
        if verbose:
            print(f"\n  ⚠️  All {len(df_f)} stocks "
                  f"RETREATING. No entries. Cash held.")
        return (df_f, pd.DataFrame(),
                'NO_STOCKS_L4', 0, 0)

    if verbose:
        print(f"\n  Health summary:")
        print(f"    ✅ RISING      : {len(rising)}")
        print(f"    ⚠️  MIXED       : {len(mixed)}")
        print(f"    ⚠️  NEUTRAL     : {len(neutral)}")
        print(f"    🚫 RETREATING  : {len(retreating)}")
        print(f"    Investable     : {investable}")

    # Phase B

    # Ask count AFTER showing list
    if n_stocks is None and verbose:
        n_stocks = ask_stock_count(
            max_available=min(
                investable, MAX_STOCK_COUNT))
    elif n_stocks is None:
        n_stocks = min(DEFAULT_STOCK_COUNT,
                       investable, MAX_STOCK_COUNT)

    n_stocks = max(1, min(
        n_stocks, investable, MAX_STOCK_COUNT))

    # Ask capital AFTER knowing count
    if capital is None and verbose:
        suggest = n_stocks * 20_000
        capital = ask_capital(
            label=label, default=suggest)
    elif capital is None:
        capital = n_stocks * 20_000

    # Warn if position size too small
    pos_size = capital / n_stocks if n_stocks > 0 else 0
    if pos_size < MIN_POSITION_RS and verbose:
        print(f"\n  ⚠️  Position size Rs{pos_size:,.0f} < "
              f"minimum Rs{MIN_POSITION_RS:,.0f}.")
        print(f"     Reduce stock count or increase capital.")

    # Dynamic sector cap
    df_invest = df_f[
        df_f['Health_Label'] != 'RETREATING'].copy()
    dyn_cap  = _dynamic_sector_cap(df_invest)
    n_sects  = df_invest['Sector'].nunique()

    if verbose:
        print(f"\n  Qualifying sectors : {n_sects}")
        print(f"  Dynamic sector cap : {dyn_cap*100:.0f}%"
              f"  ({'concentrated' if dyn_cap > 0.35 else 'diversified'})")

    # Sort for Layer 3:
    # RISING first, then MIXED, then NEUTRAL
    # Within each group by Optimizer_Score descending
    health_order = {
        'RISING'      : 0,
        'MIXED'       : 1,
        'NEUTRAL'     : 2,
        'RETREATING'  : 3,
        'UNKNOWN'     : 4,
        'NO_DATA'     : 5,
        'INSUFFICIENT': 5,
        'ERROR'       : 5,
    }
    df_f['_hsort'] = df_f['Health_Label'].map(
        health_order).fillna(4)
    df_sorted = df_f.sort_values(
        ['_hsort', 'Optimizer_Score'],
        ascending=[True, False]
    ).drop(columns=['_hsort']).reset_index(drop=True)

    # Layer 3 — size top N stocks
    df_final = compute_position_sizes(
        df_sorted, capital, n_stocks)

    # Summary
    if verbose:
        allocated = df_final[
            df_final['Shares'] > 0].copy() \
            if 'Shares' in df_final.columns \
            else pd.DataFrame()
        total_inv = (allocated['Invested'].sum()
                     if len(allocated) > 0 else 0)
        cash_idle = capital - total_inv
        idle_pct  = (cash_idle / capital * 100
                     if capital > 0 else 0)

        print(f"\n  -- Pipeline summary --")
        print(f"  Passed filters    : {len(df_f)}")
        print(f"  Investable        : {investable}")
        print(f"  Chosen count      : {n_stocks}")
        print(f"  Allocated         : {len(allocated)}")
        print(f"  Total deployed    : Rs{total_inv:,.0f} "
              f"({total_inv/capital*100:.1f}%)")
        print(f"  Cash remaining    : Rs{cash_idle:,.0f} "
              f"({idle_pct:.1f}%)")
        if idle_pct > 30:
            print(f"  ℹ️   {idle_pct:.0f}% idle — "
                  f"sector caps limited deployment.")

    status = ('OK'
              if 'Shares' in df_final.columns
              and len(df_final[df_final['Shares'] > 0]) >= 3
              else 'WARN_FEW')

    return df_f, df_final, status, capital, n_stocks


# ── DISPLAY HELPERS ───────────────────────────────────────────
SECTOR_ABBREV = {
    'Financial Services'             : 'FinSvc',
    'Metals & Mining'                : 'Metals',
    'Capital Goods'                  : 'CapGds',
    'Power'                          : 'Power',
    'Healthcare'                     : 'Health',
    'Consumer Durables'              : 'ConDur',
    'Fast Moving Consumer Goods'     : 'FMCG',
    'Automobile and Auto Components' : 'Auto',
    'Information Technology'         : 'IT',
    'Chemicals'                      : 'Chem',
    'Construction'                   : 'Const',
    'Construction Materials'         : 'CnMat',
    'Real Estate'                    : 'Realty',
    'Oil, Gas & Consumable Fuels'    : 'Oil&Gas',
    'Textiles'                       : 'Textil',
    'Pharmaceuticals & Biotech'      : 'Pharma',
    'Consumer Services'              : 'ConSvc',
    'Diversified'                    : 'Divers',
    'Media Entertainment & Pub'      : 'Media',
    'Telecommunication'              : 'Telcm',
    'Forest Materials'               : 'Forest',
    'Retailing'                      : 'Retail',
    'Insurance'                      : 'Insur',
    'Realty'                         : 'Realty',
}

HEALTH_ICON = {
    'RISING'      : '✅Rise',
    'MIXED'       : '⚠️ Mix',
    'NEUTRAL'     : '⚠️ Neu',
    'RETREATING'  : '🚫Ret',
    'NO_DATA'     : '❓ N/A',
    'INSUFFICIENT': '❓ Low',
    'UNKNOWN'     : '❓    ',
    'ERROR'       : '❌Err',
}

NOTE_SHORT = {
    'S'      : 'S',
    'Sec'    : 'Sec',
    'Tot'    : 'Tot',
    'S+Sec'  : 'S+Sc',
    'S+Tot'  : 'S+T',
    'Sec+Tot': 'Sc+T',
    'Full'   : 'Full',
    ''       : '—',
}

def _sec(s):
    return SECTOR_ABBREV.get(str(s), str(s)[:7])

def _mcap(val):
    try:
        v = float(val or 0)
    except:
        return '  —  '
    if v >= 100000: return f'{v/100000:.1f}LC'
    if v >= 1000:   return f'{v/1000:.0f}KC'
    return f'{v:.0f}Cr'

def _note(n):
    return NOTE_SHORT.get(str(n).strip(), str(n)[:5])


def _show_candidate_list(df_f):
    """
    Show ALL qualifying stocks with scores and health
    BEFORE asking capital and count.
    User sees full picture first, then decides.
    Includes R5, R20, S7 from health detail.
    """
    print(f"\n  {'='*108}")
    print(f"  ALL QUALIFYING STOCKS — {len(df_f)} stocks  "
          f"|  Date: {today_str}")
    print(f"  {'='*108}")
    print(f"  {'#':>3}  {'Symbol':<12} {'C':<2} {'Sec':<7} "
          f"{'Trend':<5} {'EMA%':>6} {'Tec':>3} "
          f"{'ML%':>3} {'Scr':>5} {'Hlth':<7} "
          f"{'R5':>4} {'R20':>4} {'MACD':>6} "
          f"{'MCap':>6} {'SR':>4}  Tier")
    print(f"  {'─'*108}")

    for i, (_, row) in enumerate(df_f.iterrows(), 1):
        sym     = str(row['Symbol'])[:12]
        cap_s   = CAP_SHORT.get(
            str(row.get('Cap Category', '')), '?')[:2]
        sector  = _sec(row.get('Sector', ''))
        trend   = ('StrUp' if 'Strong' in
                   str(row.get('Sector Trend', ''))
                   else 'Up   ')
        ema50p  = float(row.get('EMA50_Pct', 0) or 0)
        tech    = float(row.get('Tech Score_f',
                        row.get('Tech Score', 0)) or 0)
        ml_conf = float(row.get('ML_Confidence_f',
                        row.get('ML_Confidence', 0)) or 0)
        score   = float(
            row.get('Optimizer_Score', 0) or 0)
        health  = HEALTH_ICON.get(
            str(row.get('Health_Label', '')), '?    ')
        mcap    = _mcap(row.get('Market_Cap_Cr',
                        row.get('MCap', 0)))
        sec_rnk = float(
            row.get('Sector Score', 0) or 0)
        tier    = str(row.get('Tier', ''))
        tier_s  = ('T1' if 'TIER 1' in tier
                   else 'T2' if 'TIER 2' in tier
                   else '  ')

        # Extract R5, R20, MACD from health detail
        det     = str(row.get('Health_Detail', ''))
        r5      = '  — '
        r20     = '  — '
        macd_s  = '  — '
        for part in det.split('  '):
            if part.startswith('R5:'):
                r5     = part.replace('R5:', '')[:5]
            if part.startswith('R20:'):
                r20    = part.replace('R20:', '')[:5]
            if 'MACD' in part:
                macd_s = part[:6]

        print(
            f"  {i:>3}.  {sym:<12} {cap_s:<2} {sector:<7} "
            f"{trend:<5} {ema50p:>+5.1f}% "
            f"{tech:>3.0f} {ml_conf:>3.0f}% "
            f"{score:>5.1f} {health:<7} "
            f"{r5:>4} {r20:>4} {macd_s:>6}  "
            f"{mcap:>6} {sec_rnk:>4.1f}  {tier_s}")


def print_pipeline_table(df_final, capital,
                          title='OPTIMIZER OUTPUT'):
    """
    Allocation table for sized stocks.
    Sections: RISING | MIXED | NEUTRAL | RETREATING | CAPPED
    """
    SEP = "─" * 115

    def _hdr():
        print(f"\n{'='*115}")
        print(f"  {title}")
        print(f"  Capital: Rs{capital:,.0f}  |  {today_str}")
        print(f"  S=stock cap  Sec=sector cap  "
              f"Tot=total cap  Full=capital full")
        print(f"{'='*115}")
        print(
            f"  {'#':>2}  {'Symbol':<12} {'C':<2} "
            f"{'Sec':<7} {'Trnd':<5} {'EMA%':>5} "
            f"{'Tec':>3} {'ML%':>3} {'Scr':>5} "
            f"{'Hlth':<7} {'Qty':>4} "
            f"{'Invested':>9} {'I%':>4} "
            f"{'StopRef':>9} {'ATR%':>4} "
            f"{'MCap':>6} {'SR':>4}  Nt"
        )
        print(f"  {SEP}")

    def _row(serial, row):
        sym      = str(row['Symbol'])[:12]
        cap_s    = CAP_SHORT.get(
            str(row.get('Cap Category', '')), '?')[:2]
        sector   = _sec(row.get('Sector', ''))
        trend    = ('StrUp' if 'Strong' in
                    str(row.get('Sector Trend', ''))
                    else 'Up   ')
        ema50p   = float(row.get('EMA50_Pct', 0) or 0)
        tech     = float(row.get('Tech Score_f',
                         row.get('Tech Score', 0)) or 0)
        ml_conf  = float(row.get('ML_Confidence_f',
                         row.get('ML_Confidence', 0)) or 0)
        score    = float(
            row.get('Optimizer_Score', 0) or 0)
        health   = HEALTH_ICON.get(
            str(row.get('Health_Label', '')), '?    ')
        shares   = int(row.get('Shares', 0) or 0)
        invested = float(row.get('Invested', 0) or 0)
        inv_pct  = float(
            row.get('Invested_Pct', 0) or 0)
        stop_ref = float(
            row.get('Stop_Price_Ref', 0) or 0)
        atr_pct  = float(row.get('ATR_Pct', 0) or 0)
        mcap     = _mcap(row.get('Market_Cap_Cr',
                         row.get('MCap', 0)))
        sec_rnk  = float(
            row.get('Sector Score', 0) or 0)
        note     = _note(row.get('Size_Note', ''))

        print(
            f"  {serial:>2}.  {sym:<12} {cap_s:<2} "
            f"{sector:<7} {trend:<5} {ema50p:>+4.1f}% "
            f"{tech:>3.0f} {ml_conf:>3.0f}% "
            f"{score:>5.1f} {health:<7} "
            f"{shares:>4} Rs{invested:>7,.0f} "
            f"{inv_pct:>3.1f}% "
            f"Rs{stop_ref:>7.2f} {atr_pct:>3.1f}% "
            f"{mcap:>6} {sec_rnk:>4.1f}  {note}"
        )

    def _section(rows, heading):
        if len(rows) == 0:
            return
        print(f"\n  {heading}")
        print(f"  {'─'*60}")
        for i, (_, row) in enumerate(rows.iterrows(), 1):
            _row(i, row)

    if 'Shares' not in df_final.columns:
        print("  No data to display.")
        return

    _hdr()

    allocated  = df_final[
        df_final['Shares'] > 0].copy()
    capped_out = df_final[
        df_final['Shares'] == 0].copy()

    _section(
        allocated[
            allocated['Health_Label'] == 'RISING'],
        '✅ RISING — clean entry, fresh momentum')
    _section(
        allocated[
            allocated['Health_Label'] == 'MIXED'],
        '⚠️  MIXED — valid entry, some signals diverging')
    _section(
        allocated[
            allocated['Health_Label'] == 'NEUTRAL'],
        '⚠️  NEUTRAL — consolidating, valid but watch')
    _section(
        allocated[
            allocated['Health_Label'] == 'RETREATING'],
        '🚫 RETREATING — flagged, review before entering')

    # Sector summary
    if len(allocated) > 0:
        dyn_cap = _dynamic_sector_cap(allocated)
        print(f"\n  {'─'*55}")
        print(f"  SECTOR ALLOCATION  "
              f"(cap:{dyn_cap*100:.0f}%  "
              f"per-stock:{MAX_STOCK_PCT*100:.0f}%)")
        print(f"  {'─'*55}")
        sec_totals = (
            allocated.groupby('Sector')['Invested']
            .sum().sort_values(ascending=False))
        for sec, amt in sec_totals.items():
            pct  = amt / capital * 100
            bar  = '█' * int(pct * 1.5)
            warn = '  ⚠️' if pct > dyn_cap * 100 * 0.9 \
                   else ''
            print(f"  {_sec(sec):<8} {bar:<36} "
                  f"Rs{amt:>9,.0f}  {pct:>5.1f}%{warn}")

    total_inv = (allocated['Invested'].sum()
                 if len(allocated) > 0 else 0)
    print(f"\n  {'─'*55}")
    print(f"  Deployed  : Rs{total_inv:,.0f} "
          f"({total_inv/capital*100:.1f}%)")
    print(f"  Cash left : Rs{capital-total_inv:,.0f} "
          f"({(capital-total_inv)/capital*100:.1f}%)")

    if len(capped_out) > 0:
        print(f"\n  SCORED BUT NOT ALLOCATED:")
        print(f"  {'Symbol':<14} {'Score':>5}  "
              f"{'Health':<8}  Reason")
        print(f"  {'─'*48}")
        rmap = {
            'Sec' : 'Sector cap reached',
            'Full': 'Capital fully deployed',
            'S'   : 'Stock cap reached',
        }
        for _, row in capped_out.iterrows():
            n = _note(row.get('Size_Note', ''))
            print(
                f"  {str(row['Symbol']):<14} "
                f"{float(row.get('Optimizer_Score',0)):>5.1f}  "
                f"{HEALTH_ICON.get(str(row.get('Health_Label','')), '?'):<8}  "
                f"{rmap.get(n, n)}")


# ── SILENT VALIDATION ─────────────────────────────────────────
def _silent_validation():
    """
    Test all 4 layers load correctly. No prompts.
    Shows top stock with full S4/S6/S7 detail.
    """
    _cap = 200_000
    print("  Testing all 4 layers (silent)...")
    try:
        df_f = filter_candidates(tech_df, verbose=False)
        n    = len(df_f)

        if n == 0:
            print(f"  Layer 1 — 0 stocks passed")
            print(f"  ✓ No-buy guard working correctly")
            return

        df_f['Optimizer_Score'] = df_f.apply(
            compute_optimizer_score, axis=1)
        df_top  = df_f.nlargest(3, 'Optimizer_Score')
        top_sym = df_top.iloc[0]['Symbol']
        top_scr = df_top.iloc[0]['Optimizer_Score']

        df_s = compute_position_sizes(
            df_top.copy(), _cap, 3)
        h    = compute_health(str(top_sym))

        s4_val = h.get('S4_RSI', 0)
        s6_val = h.get('S6_Range20d', 0)
        s7_val = h.get('S7_MACD', 0)
        det    = h.get('Health_Detail', '')

        print(f"  Layer 1 — {n} stocks passed filters")
        print(f"  Layer 2 — top: {top_sym} "
              f"score={top_scr:.1f}")
        print(f"  Layer 3 — equal alloc "
              f"~Rs{_cap/3:,.0f}/stock")
        print(f"  Layer 4 — {top_sym}: "
              f"{h['Health_Label']} "
              f"(score={h['Health_Score']})")
        print(f"            {det}")
        print(f"  S4(RSI)     : {s4_val}")
        print(f"  S6(20d rng) : "
              f"{'bonus +1' if s6_val == 1 else 'no bonus'}")
        print(f"  S7(MACD+P)  : "
              f"{'recovery +1' if s7_val == 1 else 'penalty -2' if s7_val == -2 else 'penalty -1' if s7_val == -1 else 'neutral 0'}")

    except Exception as e:
        print(f"  ❌ Validation error: {e}")
        import traceback
        traceback.print_exc()


# ── RUN VALIDATION ────────────────────────────────────────────
print("─" * 55)
print("  Cell 3 — Loading pipeline engine")
print("─" * 55)

_silent_validation()

print(f"\n✅  Cell 3 complete")
print(f"     filter_candidates()       — Layer 1")
print(f"     compute_optimizer_score() — Layer 2")
print(f"     compute_position_sizes()  — Layer 3")
print(f"     compute_health()          — Layer 4 (7 signals)")
print(f"     run_pipeline()            — combined entry point")
print(f"     print_pipeline_table()    — allocation display")
print(f"     _show_candidate_list()    — pre-capital display")
print(f"\n  Now run Cell 4, then call run_optimizer_menu()")


───────────────────────────────────────────────────────
  Cell 3 — Loading pipeline engine
───────────────────────────────────────────────────────
  Testing all 4 layers (silent)...
  Layer 1 — 23 stocks passed filters
  Layer 2 — top: NAVA score=123.3
  Layer 3 — equal alloc ~Rs66,667/stock
  Layer 4 — NAVA: MIXED (score=1)
            Mom:+0.8%↑  R5:26%↓  E20:+1.94%↑  RSI:60→  Rec✓  R20:62%✓  MACD↓P↓(-3.6)
  S4(RSI)     : 0
  S6(20d rng) : bonus +1
  S7(MACD+P)  : penalty -2

✅  Cell 3 complete
     filter_candidates()       — Layer 1
     compute_optimizer_score() — Layer 2
     compute_position_sizes()  — Layer 3
     compute_health()          — Layer 4 (7 signals)
     run_pipeline()            — combined entry point
     print_pipeline_table()    — allocation display
     _show_candidate_list()    — pre-capital display

  Now run Cell 4, then call run_optimizer_menu()


In [15]:
# ============================================================
# DAY 15 — Portfolio Optimizer
# Cell 4 : Auto-Tester Engine (Menu Option 4)
#
# Autonomous paper trading with full control over exits
# and reinvestment. User controls fresh capital infusion.
#
# Two money pools:
#   Pool 1 — Optimizer's pool (full autonomy)
#     You give: initial capital + number of stocks
#     Optimizer: buys, exits on signals, reinvests proceeds
#     Fresh pipeline run on EVERY decision — never stale list
#
#   Pool 2 — Fresh infusion (your decision)
#     a. Pyramid existing holdings (RISING stocks only)
#     b. New stocks (fresh pipeline RIGHT NOW, not old list)
#     c. Split between pyramid and new
#     After infusion → joins optimizer pool → full autonomy
#
# Multiple named testers supported.
# Each tester has its own folder under OPT_TESTER_DIR.
#
# Sub-menu per tester:
#   a. Run tester   (exits → reinvest → log)
#   b. View positions  (live indicators + P&L + Nifty)
#   c. View log        (transaction history)
#   d. Performance     (P&L vs Nifty + chart)
#   e. Infuse capital  (pyramid / new / split)
#   f. Reset
#   g. Back
# ============================================================

import time as _time_mod

# ── TESTER FOLDER HELPERS ─────────────────────────────────────
def _tester_dir(name):
    return os.path.join(OPT_TESTER_DIR, name)

def _tester_port_file(name):
    return os.path.join(_tester_dir(name), 'portfolio.csv')

def _tester_log_file(name):
    return os.path.join(_tester_dir(name), 'log.csv')

def _tester_meta_file(name):
    return os.path.join(_tester_dir(name), 'meta.txt')

def _list_testers():
    """Return sorted list of existing tester names."""
    if not os.path.exists(OPT_TESTER_DIR):
        return []
    return sorted([
        d for d in os.listdir(OPT_TESTER_DIR)
        if os.path.isdir(os.path.join(OPT_TESTER_DIR, d))
    ])

def _load_tester_port(name):
    return load_port(
        _tester_port_file(name), TESTER_PORT_COLS)

def _save_tester_port(name, df):
    save_port(df, _tester_port_file(name))

def _load_tester_log(name):
    return load_port(
        _tester_log_file(name), TESTER_LOG_COLS)

def _save_tester_log(name, df):
    save_port(df, _tester_log_file(name))

def _load_tester_meta(name):
    meta = {
        'capital'       : 0.0,
        'total_infused' : 0.0,
        'n_stocks'      : DEFAULT_STOCK_COUNT,
        'start_date'    : '',
        'last_run'      : '',
        'last_run_ts'   : 0.0,
        'reentry_exits' : {},
    }
    fpath = _tester_meta_file(name)
    if not os.path.exists(fpath):
        return meta
    try:
        with open(fpath) as f:
            for line in f:
                line = line.strip()
                if '=' not in line:
                    continue
                k, v = line.split('=', 1)
                k, v = k.strip(), v.strip()
                if k == 'capital':
                    meta['capital'] = float(v)
                elif k == 'total_infused':
                    meta['total_infused'] = float(v)
                elif k == 'n_stocks':
                    meta['n_stocks'] = int(v)
                elif k == 'start_date':
                    meta['start_date'] = v
                elif k == 'last_run':
                    meta['last_run'] = v
                elif k == 'last_run_ts':
                    meta['last_run_ts'] = float(v)
                elif k.startswith('reentry_'):
                    sym = k.replace('reentry_', '')
                    meta['reentry_exits'][sym] = v
    except:
        pass
    return meta

def _save_tester_meta(name, meta):
    lines = [
        f"capital={meta['capital']}",
        f"total_infused={meta.get('total_infused',0)}",
        f"n_stocks={meta.get('n_stocks', DEFAULT_STOCK_COUNT)}",
        f"start_date={meta.get('start_date','')}",
        f"last_run={meta.get('last_run','')}",
        f"last_run_ts={meta.get('last_run_ts',0)}",
    ]
    for sym, dt in meta.get('reentry_exits', {}).items():
        lines.append(f"reentry_{sym}={dt}")
    os.makedirs(_tester_dir(name), exist_ok=True)
    with open(_tester_meta_file(name), 'w') as f:
        f.write('\n'.join(lines))


# ── TRANSACTION LOGGER ────────────────────────────────────────
def _log_tx(name, action, symbol, price, qty,
             ml_label, health_score, health_label,
             sector_trend, ema50_pct, opt_score,
             reason, portfolio_value):
    df_log = _load_tester_log(name)
    now    = datetime.now()
    record = {
        'Date'                  : now.strftime('%Y-%m-%d'),
        'Time'                  : now.strftime('%H:%M:%S'),
        'Action'                : action,
        'Symbol'                : symbol,
        'Price'                 : round(float(price), 2),
        'Quantity'              : int(qty),
        'Amount'                : round(float(price)*int(qty), 2),
        'ML_Label'              : ml_label,
        'Health_Score'          : health_score,
        'Health_Label'          : health_label,
        'Sector_Trend'          : sector_trend,
        'EMA50_Pct'             : round(float(ema50_pct), 1),
        'Optimizer_Score'       : round(float(opt_score), 1),
        'Reason'                : reason,
        'Portfolio_Value_After' : round(float(portfolio_value), 2),
        'Tester_Name'           : name,
    }
    df_log = pd.concat(
        [df_log, pd.DataFrame([record])],
        ignore_index=True)
    _save_tester_log(name, df_log)


# ── RE-ENTRY ELIGIBILITY ──────────────────────────────────────
def _reentry_ok(meta, symbol):
    """True if stock is eligible for re-entry."""
    reentry_exits = meta.get('reentry_exits', {})
    if symbol not in reentry_exits:
        return True
    exit_dt  = pd.Timestamp(reentry_exits[symbol])
    today_dt = pd.Timestamp(today_str)
    gap_days = (today_dt - exit_dt).days
    return gap_days >= REENTRY_GAP_DAYS


# ── TESTER INITIALISATION ─────────────────────────────────────
def _init_tester(name):
    """
    First run for a named tester.
    Asks capital + stock count, runs full pipeline,
    buys top N stocks, writes all files.
    """
    print(f"\n  AUTO-TESTER '{name}' — INITIALISATION")
    print("  " + "─"*50)
    print("  Runs pipeline RIGHT NOW on fresh data.")
    print("  Optimizer controls entries/exits/reinvestment.")
    print("  You control fresh capital infusion only.\n")

    # Run pipeline (asks count then capital inside)
    df_all, df_final, status, capital, n_stocks = \
        run_pipeline(label=f"tester '{name}' init")

    meta = {
        'capital'       : capital,
        'total_infused' : capital,
        'n_stocks'      : n_stocks,
        'start_date'    : today_str,
        'last_run'      : today_str,
        'last_run_ts'   : _time_mod.time(),
        'reentry_exits' : {},
    }
    os.makedirs(_tester_dir(name), exist_ok=True)
    _save_tester_log(
        name, pd.DataFrame(columns=TESTER_LOG_COLS))

    if status in ('NO_STOCKS_L1', 'NO_BUY',
                  'NO_STOCKS_L4'):
        print(f"\n  ⚠️  No qualifying stocks today.")
        print(f"     Tester '{name}' created with "
              f"Rs{capital:,.0f} in cash.")
        print(f"     Run again after next data update.")
        _save_tester_port(
            name,
            pd.DataFrame(columns=TESTER_PORT_COLS))
        _save_tester_meta(name, meta)
        return

    # Buy RISING + MIXED + NEUTRAL (not RETREATING)
    allocated = df_final[
        (df_final['Shares'] > 0) &
        (~df_final['Health_Label'].isin(['RETREATING']))
    ].copy() if 'Shares' in df_final.columns \
        else pd.DataFrame()

    if len(allocated) == 0:
        print(f"\n  ⚠️  All allocated stocks RETREATING.")
        print(f"     No entries. Cash held.")
        _save_tester_port(
            name,
            pd.DataFrame(columns=TESTER_PORT_COLS))
        _save_tester_meta(name, meta)
        return

    print_pipeline_table(
        df_final, capital,
        title=f"TESTER '{name.upper()}' — INITIAL BUY")

    port_records  = []
    total_invested = 0
    now_str        = datetime.now().strftime('%H:%M:%S')

    for _, row in allocated.iterrows():
        sym    = str(row['Symbol'])
        price  = float(row.get('Current Price', 0) or 0)
        shares = int(row.get('Shares', 0) or 0)
        inv    = float(row.get('Invested', 0) or 0)
        if shares == 0 or price <= 0:
            continue

        ml_pred, ml_conf = get_current_ml(sym)
        trend            = get_current_trend(sym)
        ema50p           = compute_ema50_pct(sym)
        score            = float(
            row.get('Optimizer_Score', 0) or 0)
        h_lbl  = str(row.get('Health_Label', ''))
        h_scr  = int(row.get('Health_Score', 0) or 0)
        stop_r = float(
            row.get('Stop_Price_Ref', 0) or 0)

        port_records.append({
            'Symbol'                  : sym,
            'Entry_Price'             : price,
            'Entry_Date'              : today_str,
            'Entry_Time'              : now_str,
            'Quantity'                : shares,
            'Cap_Category'            : str(
                row.get('Cap Category', '')),
            'Invested'                : inv,
            'ML_At_Entry'             : ml_pred,
            'Sector_Trend_At_Entry'   : trend,
            'EMA50_Pct_At_Entry'      : ema50p,
            'Optimizer_Score_At_Entry': score,
            'Health_At_Entry'         : h_lbl,
            'Stop_Price_Ref'          : stop_r,
            'Last_Checked'            : today_str,
            'Status'                  : 'HOLD',
            'Tester_Name'             : name,
        })
        total_invested += inv

        reason = (
            f"INIT BUY | Score:{score:.1f} | "
            f"EMA50%:{ema50p:+.1f}% | "
            f"ML:{ml_pred} {ml_conf:.0f}% | "
            f"Health:{h_lbl}({h_scr})"
        )
        _log_tx(name, 'BUY', sym, price, shares,
                ml_pred, h_scr, h_lbl,
                trend, ema50p, score,
                reason, total_invested)

    _save_tester_port(name, pd.DataFrame(port_records))
    _save_tester_meta(name, meta)

    print(f"\n  ✅ Tester '{name}' started")
    print(f"     Capital   : Rs{capital:,.0f}")
    print(f"     Positions : {len(port_records)} stocks")
    print(f"     Invested  : Rs{total_invested:,.0f} "
          f"({total_invested/capital*100:.1f}%)")
    print(f"     Cash held : "
          f"Rs{capital-total_invested:,.0f}")


# ── TESTER UPDATE (runs on every data refresh) ────────────────
def _run_tester_update(name):
    """
    Called every time after run_weekly updates data.

    Step 1: Check exits — ML downgrade or Layer 4 ≤ -3
    Step 2: Auto-redeploy proceeds with FRESH pipeline
            (never stale list — always current best stocks)
    Step 3: Check if new stocks appeared since last run
    Step 4: Log all decisions
    """
    meta    = _load_tester_meta(name)
    df_port = _load_tester_port(name)
    capital = meta['capital']
    n_stocks = meta.get('n_stocks', DEFAULT_STOCK_COUNT)

    print(f"\n  TESTER '{name}' UPDATE — {today_str}")
    print(f"  Capital: Rs{capital:,.0f}  "
          f"| Positions: {len(df_port)}  "
          f"| Target: {n_stocks} stocks")
    print("  " + "─"*50)

    reentry_exits = meta.get('reentry_exits', {})
    exits         = []
    holds         = []
    total_freed   = 0.0

    # ── Step 1: Check exits ───────────────────────────────
    print(f"\n  Checking {len(df_port)} holdings...")

    for _, row in df_port.iterrows():
        sym       = str(row['Symbol'])
        entry_px  = float(row.get('Entry_Price', 0) or 0)
        qty       = int(row.get('Quantity', 0) or 0)
        ml_entry  = str(row.get(
            'ML_At_Entry', 'Bullish Continual'))

        curr_px            = get_current_price(sym)
        curr_ml, curr_conf = get_current_ml(sym)
        curr_trend         = get_current_trend(sym)
        curr_ema50         = compute_ema50_pct(sym)
        h                  = compute_health(sym)
        h_scr              = h['Health_Score']
        h_lbl              = h['Health_Label']
        pl_pct             = (
            (curr_px - entry_px) / entry_px * 100
            if entry_px > 0 and curr_px > 0 else 0)

        exit_reason = None

        # Primary: ML degradation
        if (ml_entry == 'Bullish Continual'
                and curr_ml in EXIT_FLAG_LABELS):
            exit_reason = (
                f"EXIT-ML | {ml_entry}→{curr_ml} | "
                f"P&L:{pl_pct:+.1f}%")

        # Secondary: Layer 4 score ≤ threshold
        elif h_scr <= HEALTH_AUTO_EXIT_THRESHOLD:
            exit_reason = (
                f"EXIT-L4 | Health:{h_lbl}({h_scr}) | "
                f"{h['Health_Detail'][:40]} | "
                f"P&L:{pl_pct:+.1f}%")

        # Lock gains: big profit + ML reversed to bearish
        elif (curr_ml in ['Bearish Continual', 'Bearish']
              and pl_pct > 10):
            exit_reason = (
                f"LOCK-GAINS | ML:{curr_ml} | "
                f"P&L:+{pl_pct:.1f}%")

        if exit_reason:
            freed = (curr_px * qty if curr_px > 0
                     else entry_px * qty)
            total_freed += freed
            exits.append({
                'sym'         : sym,
                'curr_px'     : curr_px,
                'qty'         : qty,
                'freed'       : freed,
                'pl_pct'      : pl_pct,
                'exit_reason' : exit_reason,
                'curr_ml'     : curr_ml,
                'h_scr'       : h_scr,
                'h_lbl'       : h_lbl,
                'curr_trend'  : curr_trend,
                'curr_ema50'  : curr_ema50,
            })
            print(f"  🔴 EXIT  {sym:<14} "
                  f"Rs{entry_px:.2f}→Rs{curr_px:.2f}  "
                  f"{pl_pct:+.1f}%  |  "
                  f"{exit_reason[:55]}")
        else:
            holds.append(row)
            print(f"  ✅ HOLD  {sym:<14} "
                  f"Rs{entry_px:.2f}→Rs{curr_px:.2f}  "
                  f"{pl_pct:+.1f}%  "
                  f"ML:{curr_ml[:14]}  "
                  f"Health:{h_lbl}({h_scr})")

    # Execute exits — log and record reentry gap
    held_val = sum(
        get_current_price(str(r.get('Symbol', ''))) *
        int(r.get('Quantity', 0))
        for r in holds
    ) if holds else 0

    if exits:
        port_val_after = held_val + total_freed
        for ex in exits:
            _log_tx(name,
                    'SELL', ex['sym'],
                    ex['curr_px'], ex['qty'],
                    ex['curr_ml'],
                    ex['h_scr'], ex['h_lbl'],
                    ex['curr_trend'],
                    ex['curr_ema50'], 0,
                    ex['exit_reason'],
                    port_val_after)
            reentry_exits[ex['sym']] = today_str

        meta['reentry_exits'] = reentry_exits
        print(f"\n  Exited {len(exits)} stock(s). "
              f"Freed: Rs{total_freed:,.0f}")

    # Update portfolio — remove exited stocks
    df_port = pd.DataFrame(holds) \
        if holds else pd.DataFrame(
        columns=TESTER_PORT_COLS)

    # ── Step 2: Auto-redeploy proceeds ───────────────────
    free_cash = total_freed
    slots_open = n_stocks - len(df_port)

    if free_cash > 0 and slots_open > 0:
        held_syms = set(df_port['Symbol'].tolist()) \
            if len(df_port) > 0 else set()

        # Ineligible = held OR in reentry gap
        ineligible = held_syms | {
            sym for sym in reentry_exits
            if not _reentry_ok(meta, sym)
        }

        print(f"\n  Auto-redeploying Rs{free_cash:,.0f} "
              f"into {slots_open} slot(s)...")
        print(f"  Running fresh pipeline "
              f"(excluding {len(ineligible)} "
              f"held/recent stocks)...")

        # Fresh pipeline — current best, not stale list
        total_port_val = held_val + free_cash
        alloc_per_slot = total_port_val / n_stocks

        _df_all, df_new, new_status, _cap, _n = \
            run_pipeline(
                label=f"tester '{name}' redeploy",
                exclude_symbols=ineligible,
                verbose=False,
                capital=free_cash,
                n_stocks=slots_open)

        if new_status not in (
                'NO_STOCKS_L1', 'NO_BUY', 'NO_STOCKS_L4'):
            buyable = df_new[
                (df_new['Shares'] > 0) &
                (~df_new['Health_Label'].isin(
                    ['RETREATING']))
            ].copy() if 'Shares' in df_new.columns \
                else pd.DataFrame()

            new_entries = []
            now_str     = datetime.now().strftime(
                '%H:%M:%S')

            for _, row in buyable.iterrows():
                sym    = str(row['Symbol'])
                price  = float(
                    row.get('Current Price', 0) or 0)
                shares = int(row.get('Shares', 0) or 0)
                inv    = float(
                    row.get('Invested', 0) or 0)
                if shares == 0 or price <= 0:
                    continue

                ml_pred, ml_conf = get_current_ml(sym)
                trend  = get_current_trend(sym)
                ema50p = compute_ema50_pct(sym)
                score  = float(
                    row.get('Optimizer_Score', 0) or 0)
                h_lbl  = str(
                    row.get('Health_Label', ''))
                h_scr  = int(
                    row.get('Health_Score', 0) or 0)
                stop_r = float(
                    row.get('Stop_Price_Ref', 0) or 0)

                new_entries.append({
                    'Symbol'                  : sym,
                    'Entry_Price'             : price,
                    'Entry_Date'              : today_str,
                    'Entry_Time'              : now_str,
                    'Quantity'                : shares,
                    'Cap_Category'            : str(
                        row.get('Cap Category', '')),
                    'Invested'                : inv,
                    'ML_At_Entry'             : ml_pred,
                    'Sector_Trend_At_Entry'   : trend,
                    'EMA50_Pct_At_Entry'      : ema50p,
                    'Optimizer_Score_At_Entry': score,
                    'Health_At_Entry'         : h_lbl,
                    'Stop_Price_Ref'          : stop_r,
                    'Last_Checked'            : today_str,
                    'Status'                  : 'HOLD',
                    'Tester_Name'             : name,
                })

                reason = (
                    f"AUTO-BUY | Score:{score:.1f} | "
                    f"EMA50%:{ema50p:+.1f}% | "
                    f"ML:{ml_pred} {ml_conf:.0f}% | "
                    f"Health:{h_lbl}({h_scr})")

                port_val = held_val + inv
                _log_tx(name, 'BUY', sym,
                        price, shares,
                        ml_pred, h_scr, h_lbl,
                        trend, ema50p, score,
                        reason, port_val)

                print(f"  🟢 BUY   {sym:<14} "
                      f"{shares}×Rs{price:.2f}"
                      f"=Rs{inv:,.0f}  "
                      f"Health:{h_lbl}({h_scr})")

            if new_entries:
                df_port = pd.concat(
                    [df_port,
                     pd.DataFrame(new_entries)],
                    ignore_index=True)
        else:
            print(f"  No new qualifying stocks — "
                  f"cash Rs{free_cash:,.0f} held.")

    elif free_cash > 0 and slots_open == 0:
        print(f"\n  Portfolio at max {n_stocks} stocks. "
              f"Rs{free_cash:,.0f} held as cash.")
        print(f"  Consider infusing to pyramid "
              f"or exiting to create slots.")

    # ── Step 3: Check new stocks in universe ─────────────
    # Were there new stocks qualifying this week?
    held_syms_now = set(df_port['Symbol'].tolist()) \
        if len(df_port) > 0 else set()
    ineligible_now = held_syms_now | {
        sym for sym in reentry_exits
        if not _reentry_ok(meta, sym)
    }

    _df_check, _, chk_status, _, _ = run_pipeline(
        label='new stock check',
        exclude_symbols=ineligible_now,
        verbose=False,
        capital=1,      # dummy — just checking counts
        n_stocks=1)

    if (chk_status not in (
            'NO_STOCKS_L1', 'NO_BUY')
            and len(_df_check) > 0):
        new_qualify = len(_df_check)
        if new_qualify > 0:
            print(f"\n  ℹ️   {new_qualify} additional stock(s) "
                  f"qualify but portfolio is "
                  f"{'full' if slots_open <= 0 else 'has cash'}.")
            if slots_open <= 0:
                print(f"     Use Infuse Capital (option e) "
                      f"to add more positions.")
            top3 = _df_check.head(3)
            for _, r in top3.iterrows():
                h = compute_health(str(r['Symbol']))
                print(f"     → {str(r['Symbol']):<14} "
                      f"Score:{float(r.get('Optimizer_Score',0)):.1f}  "
                      f"Health:{h['Health_Label']}  "
                      f"EMA50%:{float(r.get('EMA50_Pct',0)):+.1f}%")

    # Save everything
    _save_tester_port(name, df_port)
    meta['last_run']    = today_str
    meta['last_run_ts'] = _time_mod.time()
    _save_tester_meta(name, meta)

    print(f"\n  ── Update complete ───────────────────")
    print(f"  Exited  : {len(exits)} stocks")
    print(f"  Entered : "
          f"{len(df_port) - len(holds)} stocks")
    print(f"  Holding : {len(df_port)} stocks")


# ── VIEW CURRENT POSITIONS ────────────────────────────────────
def _view_positions(name):
    """
    Live indicators on each holding.
    Re-runs Layer 4 fresh — always current, not from entry.
    Shows P&L and Nifty comparison.
    """
    meta    = _load_tester_meta(name)
    df_port = _load_tester_port(name)
    capital = meta['capital']

    print(f"\n  TESTER '{name}' — POSITIONS")
    print(f"  Started:{meta.get('start_date','?')}  "
          f"Capital:Rs{capital:,.0f}  "
          f"Last run:{meta.get('last_run','?')}")
    print(f"  {'─'*105}")

    if len(df_port) == 0:
        print("  No open positions.")
        return

    print(
        f"  {'#':>2}  {'Symbol':<12} "
        f"{'Entry':>8} {'Curr':>8} {'P&L%':>6}  "
        f"{'ML':<18} {'Hlth':<10} "
        f"{'EMA50%':>7} {'Trend':<14} "
        f"{'Days':>5}  Flags"
    )
    print(f"  {'─'*105}")

    total_inv  = 0
    total_curr = 0
    earliest   = today_str

    for i, (_, row) in enumerate(df_port.iterrows(), 1):
        sym      = str(row['Symbol'])
        entry_px = float(row.get('Entry_Price', 0) or 0)
        qty      = int(row.get('Quantity', 0) or 0)
        invested = entry_px * qty
        total_inv += invested

        curr_px  = get_current_price(sym)
        curr_px  = curr_px if curr_px > 0 else entry_px
        curr_val = curr_px * qty
        total_curr += curr_val
        pl_pct   = ((curr_val - invested) / invested * 100
                    if invested > 0 else 0)

        curr_ml, curr_conf = get_current_ml(sym)
        curr_trend         = get_current_trend(sym)
        curr_ema50         = compute_ema50_pct(sym)
        h                  = compute_health(sym)
        h_lbl              = h['Health_Label']
        h_scr              = h['Health_Score']

        entry_date = str(row.get('Entry_Date', today_str))
        try:
            days_held = (pd.Timestamp(today_str) -
                         pd.Timestamp(entry_date)).days
            if entry_date < earliest:
                earliest = entry_date
        except:
            days_held = 0

        # Live flags
        flags = []
        ml_entry = str(row.get(
            'ML_At_Entry', 'Bullish Continual'))
        if (ml_entry == 'Bullish Continual'
                and curr_ml in EXIT_FLAG_LABELS):
            flags.append('🔴EXIT-ML')
        if h_scr <= HEALTH_AUTO_EXIT_THRESHOLD:
            flags.append('🔴EXIT-L4')
        if (curr_ml in ['Bearish Continual']
                and pl_pct > 10):
            flags.append('🏆LOCK')

        h_str  = (f"{HEALTH_ICON.get(h_lbl,'?')}"
                  f"({h_scr})")
        flag_s = ' '.join(flags)

        print(
            f"  {i:>2}.  {sym:<12} "
            f"Rs{entry_px:>7.2f} "
            f"Rs{curr_px:>7.2f} {pl_pct:>+5.1f}%  "
            f"{curr_ml[:16]:<18} "
            f"{h_str:<10} "
            f"{curr_ema50:>+6.1f}% "
            f"{short_trend(curr_trend)[:13]:<14} "
            f"{days_held:>5}d"
            + (f"  {flag_s}" if flag_s else ""))
        if h['Health_Detail']:
            print(f"       └─ {h['Health_Detail']}")

    # Totals
    total_pl     = total_curr - total_inv
    total_pl_pct = (total_pl / total_inv * 100
                    if total_inv > 0 else 0)

    print(f"\n  {'─'*60}")
    print(f"  Invested  : Rs{total_inv:,.0f}")
    print(f"  Current   : Rs{total_curr:,.0f}")
    print(f"  P&L       : Rs{total_pl:+,.0f} "
          f"({total_pl_pct:+.1f}%)")

    # Nifty comparison
    nifty = fetch_nifty_return(earliest)
    if nifty:
        n_s, n_c, n_p = nifty
        alpha = total_pl_pct - n_p
        print(f"  Nifty 50  : {n_p:+.1f}%  "
              f"({n_s:.0f}→{n_c:.0f})")
        print(f"  Alpha     : {alpha:+.1f}%  "
              f"{'✅ outperforming' if alpha > 0 else '❌ underperforming'}")


# ── VIEW LOG ──────────────────────────────────────────────────
def _view_log(name, last_n=20):
    df_log = _load_tester_log(name)
    if len(df_log) == 0:
        print("  No transactions yet.")
        return

    show_n = min(last_n, len(df_log))
    print(f"\n  TESTER '{name}' — LOG "
          f"(last {show_n} of {len(df_log)})")
    print(f"  {'─'*100}")
    print(
        f"  {'Date':<12} {'Time':<8} {'Act':<6} "
        f"{'Symbol':<12} {'Price':>8} {'Qty':>5} "
        f"{'Amount':>10}  Reason"
    )
    print(f"  {'─'*100}")

    for _, row in df_log.tail(last_n).iterrows():
        action = str(row.get('Action', ''))
        icon   = {'BUY'   : '🟢',
                  'SELL'  : '🔴',
                  'INFUSE': '💰',
                  'PYRAMID':'📈'}.get(action, '  ')
        price  = float(row.get('Price', 0) or 0)
        qty    = int(row.get('Quantity', 0) or 0)
        amt    = float(row.get('Amount', 0) or 0)
        reason = str(row.get('Reason', ''))[:55]
        print(
            f"  {str(row.get('Date','')):<12} "
            f"{str(row.get('Time','')):<8} "
            f"{icon}{action:<5} "
            f"{str(row.get('Symbol','')):<12} "
            f"Rs{price:>7.2f} {qty:>5} "
            f"Rs{amt:>9,.0f}  {reason}")

    try:
        raw = input(
            f"\n  Show more? Enter N or Enter to skip: "
        ).strip()
        if raw:
            _view_log(name, int(raw))
    except:
        pass


# ── PERFORMANCE VIEW ──────────────────────────────────────────
def _view_performance(name):
    """Full P&L vs Nifty 50. Realised + unrealised."""
    meta    = _load_tester_meta(name)
    df_log  = _load_tester_log(name)
    df_port = _load_tester_port(name)
    capital = meta['capital']

    print(f"\n  TESTER '{name}' — PERFORMANCE")
    print(f"  {'─'*60}")
    print(f"  Started      : {meta.get('start_date','?')}")
    print(f"  Last run     : {meta.get('last_run','?')}")
    print(f"  Initial cap  : Rs{capital:,.0f}")
    print(f"  Total infused: "
          f"Rs{meta.get('total_infused',capital):,.0f}")

    sells = df_log[df_log['Action'] == 'SELL'].copy() \
        if len(df_log) > 0 else pd.DataFrame()
    buys  = df_log[df_log['Action'] == 'BUY'].copy() \
        if len(df_log) > 0 else pd.DataFrame()

    # Realised P&L
    realised_pl    = 0.0
    winning_trades = 0
    losing_trades  = 0
    trade_pls      = []

    for _, sell_row in sells.iterrows():
        sym      = str(sell_row.get('Symbol', ''))
        sell_px  = float(sell_row.get('Price', 0) or 0)
        qty      = int(sell_row.get('Quantity', 0) or 0)
        sym_buys = buys[buys['Symbol'] == sym]
        if len(sym_buys) > 0:
            buy_px   = float(
                sym_buys.iloc[0].get('Price', sell_px))
            trade_pl = (sell_px - buy_px) * qty
            trade_pct = ((sell_px - buy_px) / buy_px * 100
                         if buy_px > 0 else 0)
            realised_pl += trade_pl
            trade_pls.append((sym, trade_pct, trade_pl))
            if trade_pl > 0:
                winning_trades += 1
            else:
                losing_trades += 1

    # Unrealised P&L
    unrealised_pl  = 0.0
    total_invested = 0.0
    for _, row in df_port.iterrows():
        ep  = float(row.get('Entry_Price', 0) or 0)
        qty = int(row.get('Quantity', 0) or 0)
        cp  = get_current_price(str(row.get('Symbol','')))
        cp  = cp if cp > 0 else ep
        unrealised_pl  += (cp - ep) * qty
        total_invested += ep * qty

    total_infused = meta.get('total_infused', capital)
    total_pl      = realised_pl + unrealised_pl
    total_pct     = (total_pl / total_infused * 100
                     if total_infused > 0 else 0)
    total_trades  = winning_trades + losing_trades
    win_rate      = (winning_trades / total_trades * 100
                     if total_trades > 0 else 0)

    print(f"\n  ── P&L ─────────────────────────────────")
    print(f"  Realised P&L    : Rs{realised_pl:+,.0f}")
    print(f"  Unrealised P&L  : Rs{unrealised_pl:+,.0f}")
    print(f"  Total P&L       : Rs{total_pl:+,.0f} "
          f"({total_pct:+.1f}%)")
    print(f"  Open positions  : {len(df_port)} stocks")

    print(f"\n  ── Trades ──────────────────────────────")
    print(f"  Total buys   : {len(buys)}")
    print(f"  Total sells  : {len(sells)}")
    print(f"  Closed trades: {total_trades}  "
          f"Win:{winning_trades}  "
          f"Loss:{losing_trades}  "
          f"Win rate:{win_rate:.0f}%")

    if trade_pls:
        best  = max(trade_pls, key=lambda x: x[1])
        worst = min(trade_pls, key=lambda x: x[1])
        print(f"  Best trade   : "
              f"{best[0]} {best[1]:+.1f}%")
        print(f"  Worst trade  : "
              f"{worst[0]} {worst[1]:+.1f}%")

    # Nifty comparison
    start_date = meta.get('start_date', today_str)
    nifty      = fetch_nifty_return(start_date)
    if nifty:
        n_s, n_c, n_p = nifty
        alpha = total_pct - n_p
        print(f"\n  ── vs Nifty 50 (from {start_date}) ──")
        print(f"  Portfolio    : {total_pct:+.1f}%")
        print(f"  Nifty 50     : {n_p:+.1f}%  "
              f"({n_s:.0f}→{n_c:.0f})")
        print(f"  Alpha        : {alpha:+.1f}%  "
              f"{'✅ Outperforming' if alpha > 0 else '❌ Underperforming'}")

    # Chart
    if (len(df_port) > 0
            and input(
                "\n  Generate chart vs Nifty? (y/n): "
            ).strip().lower() == 'y'):
        plot_portfolio_vs_nifty(
            f"tester_{name}", df_port, start_date)


# ── CAPITAL INFUSION ──────────────────────────────────────────
def _infuse_capital(name):
    """
    Fresh capital deployment — three modes:
      a. Pyramid existing holdings (RISING only)
      b. New stocks (fresh pipeline RIGHT NOW)
      c. Split between a and b
    """
    meta    = _load_tester_meta(name)
    df_port = _load_tester_port(name)
    capital = meta['capital']

    try:
        raw = input(
            f"\n  Amount to infuse (Rs): "
        ).strip().replace(',', '')
        infusion = float(raw)
        if infusion <= 0:
            raise ValueError
    except:
        print("  Invalid amount.")
        return

    print(f"\n  Rs{infusion:,.0f} to deploy.")
    print("\n  How to deploy?")
    print("    a. Pyramid existing holdings "
          "(add to RISING stocks within 15% cap)")
    print("    b. New stocks "
          "(fresh pipeline RIGHT NOW — current best unowned)")
    print("    c. Split (specify how much each)")
    mode = input("\n  Choice (a/b/c): ").strip().lower()

    pyramid_amt = 0.0
    new_amt     = 0.0

    if mode == 'a':
        pyramid_amt = infusion
    elif mode == 'b':
        new_amt = infusion
    elif mode == 'c':
        try:
            raw_p = input(
                f"  Amount for pyramid "
                f"(Rs, rest goes to new): "
            ).strip().replace(',', '')
            pyramid_amt = float(raw_p)
            new_amt     = infusion - pyramid_amt
            if pyramid_amt < 0 or new_amt < 0:
                raise ValueError
        except:
            print("  Invalid split. Cancelled.")
            return
    else:
        print("  Invalid choice. Cancelled.")
        return

    total_port_val = (
        sum(get_current_price(str(r.get('Symbol',''))) *
            int(r.get('Quantity', 0))
            for _, r in df_port.iterrows())
        + infusion)
    new_n_stocks = meta.get('n_stocks', DEFAULT_STOCK_COUNT)

    # ── Pyramid mode ─────────────────────────────────────
    if pyramid_amt > 0 and len(df_port) > 0:
        print(f"\n  PYRAMID — Rs{pyramid_amt:,.0f}")
        print(f"  Only RISING stocks eligible. "
              f"Capped at {MAX_STOCK_PCT*100:.0f}% of "
              f"total portfolio Rs{total_port_val:,.0f} "
              f"= Rs{total_port_val*MAX_STOCK_PCT:,.0f} "
              f"per stock\n")

        pyramid_candidates = []
        for _, row in df_port.iterrows():
            sym     = str(row['Symbol'])
            h       = compute_health(sym)
            curr_px = get_current_price(sym)
            entry_px = float(
                row.get('Entry_Price', 0) or 0)
            qty      = int(row.get('Quantity', 0) or 0)
            curr_val = curr_px * qty
            pl_pct   = ((curr_px - entry_px) /
                        entry_px * 100
                        if entry_px > 0 else 0)
            max_total = total_port_val * MAX_STOCK_PCT
            room      = max(0, max_total - curr_val)

            if h['Health_Label'] == 'RISING' and room > 0:
                pyramid_candidates.append({
                    'Symbol'   : sym,
                    'Curr_Px'  : curr_px,
                    'P&L%'     : pl_pct,
                    'Health'   : h['Health_Label'],
                    'H_Score'  : h['Health_Score'],
                    'Room'     : room,
                    'H_Detail' : h['Health_Detail'],
                })

        if not pyramid_candidates:
            print("  No RISING stocks eligible "
                  "for pyramiding.")
            new_amt += pyramid_amt
            pyramid_amt = 0
        else:
            print(f"  {'#':>2}  {'Symbol':<14} "
                  f"{'CurrPx':>8} {'P&L%':>6} "
                  f"{'Health':<10} {'CanAdd':>10}")
            print("  " + "─"*55)
            for i, c in enumerate(
                    pyramid_candidates, 1):
                print(
                    f"  {i:>2}.  "
                    f"{c['Symbol']:<14} "
                    f"Rs{c['Curr_Px']:>7.2f} "
                    f"{c['P&L%']:>+5.1f}%  "
                    f"{c['Health']:<10} "
                    f"Rs{c['Room']:>9,.0f}")

            raw_pick = input(
                "\n  Which to pyramid? "
                "(comma-separated numbers or 'all'): "
            ).strip().lower()

            if raw_pick == 'all':
                picks = list(range(
                    len(pyramid_candidates)))
            else:
                try:
                    picks = [
                        int(x.strip()) - 1
                        for x in raw_pick.split(',')]
                except:
                    picks = []

            selected = [
                pyramid_candidates[i]
                for i in picks
                if 0 <= i < len(pyramid_candidates)]

            if selected:
                alloc_each = pyramid_amt / len(selected)
                for c in selected:
                    amt    = min(alloc_each, c['Room'])
                    shares = max(1, int(
                        amt / c['Curr_Px']))
                    actual = round(
                        shares * c['Curr_Px'], 2)
                    sym    = c['Symbol']

                    curr_ml, _ = get_current_ml(sym)
                    trend      = get_current_trend(sym)
                    ema50p     = compute_ema50_pct(sym)
                    opt_score  = float(
                        df_port[df_port['Symbol'] == sym
                        ].iloc[0].get(
                            'Optimizer_Score_At_Entry',
                            0) or 0)

                    reason = (
                        f"PYRAMID | "
                        f"Health:{c['Health']}({c['H_Score']}) | "
                        f"P&L:{c['P&L%']:+.1f}% | "
                        f"Rs{actual:,.0f} added")

                    _log_tx(name, 'PYRAMID',
                            sym, c['Curr_Px'], shares,
                            curr_ml, c['H_Score'],
                            c['Health'], trend,
                            ema50p, opt_score,
                            reason, total_port_val)

                    print(f"  📈 PYRAMID {sym:<14} "
                          f"+{shares}sh "
                          f"×Rs{c['Curr_Px']:.2f}"
                          f"=Rs{actual:,.0f}")

    # ── New stocks mode ───────────────────────────────────
    if new_amt > 0:
        print(f"\n  NEW STOCKS — Rs{new_amt:,.0f}")
        print("  Running fresh pipeline RIGHT NOW...")

        held_syms = set(df_port['Symbol'].tolist()) \
            if len(df_port) > 0 else set()
        reentry_exits = meta.get('reentry_exits', {})
        ineligible    = held_syms | {
            sym for sym in reentry_exits
            if not _reentry_ok(meta, sym)
        }

        _df_all, df_new, new_status, _c, _n = \
            run_pipeline(
                label='infusion — new stocks',
                exclude_symbols=ineligible,
                verbose=True,
                capital=new_amt,
                n_stocks=None)  # user chooses

        if new_status not in (
                'NO_STOCKS_L1', 'NO_BUY', 'NO_STOCKS_L4'):
            print_pipeline_table(
                df_new, new_amt,
                title='INFUSION — NEW STOCK CANDIDATES')

            buyable = df_new[
                (df_new['Shares'] > 0) &
                (~df_new['Health_Label'].isin(
                    ['RETREATING']))
            ].copy() if 'Shares' in df_new.columns \
                else pd.DataFrame()

            now_str = datetime.now().strftime('%H:%M:%S')
            new_records = []

            for _, row in buyable.iterrows():
                sym    = str(row['Symbol'])
                price  = float(
                    row.get('Current Price', 0) or 0)
                shares = int(row.get('Shares', 0) or 0)
                inv    = float(
                    row.get('Invested', 0) or 0)
                if shares == 0 or price <= 0:
                    continue

                ml_pred, ml_conf = get_current_ml(sym)
                trend  = get_current_trend(sym)
                ema50p = compute_ema50_pct(sym)
                score  = float(
                    row.get('Optimizer_Score', 0) or 0)
                h_lbl  = str(
                    row.get('Health_Label', ''))
                h_scr  = int(
                    row.get('Health_Score', 0) or 0)
                stop_r = float(
                    row.get('Stop_Price_Ref', 0) or 0)

                new_records.append({
                    'Symbol'                  : sym,
                    'Entry_Price'             : price,
                    'Entry_Date'              : today_str,
                    'Entry_Time'              : now_str,
                    'Quantity'                : shares,
                    'Cap_Category'            : str(
                        row.get('Cap Category', '')),
                    'Invested'                : inv,
                    'ML_At_Entry'             : ml_pred,
                    'Sector_Trend_At_Entry'   : trend,
                    'EMA50_Pct_At_Entry'      : ema50p,
                    'Optimizer_Score_At_Entry': score,
                    'Health_At_Entry'         : h_lbl,
                    'Stop_Price_Ref'          : stop_r,
                    'Last_Checked'            : today_str,
                    'Status'                  : 'HOLD',
                    'Tester_Name'             : name,
                })

                reason = (
                    f"INFUSE-BUY | Score:{score:.1f} | "
                    f"EMA50%:{ema50p:+.1f}% | "
                    f"ML:{ml_pred} {ml_conf:.0f}% | "
                    f"Health:{h_lbl}({h_scr})")

                _log_tx(name, 'BUY', sym,
                        price, shares,
                        ml_pred, h_scr, h_lbl,
                        trend, ema50p, score,
                        reason, total_port_val)

            if new_records:
                df_port = pd.concat(
                    [df_port,
                     pd.DataFrame(new_records)],
                    ignore_index=True)
                _save_tester_port(name, df_port)
        else:
            print(f"  No new qualifying stocks now.")

    # Update meta with new capital total
    meta['total_infused'] = (
        meta.get('total_infused', capital) + infusion)
    meta['last_run_ts'] = _time_mod.time()

    _log_tx(name, 'INFUSE', '—', 0, 0,
            '—', 0, '—', '—', 0, 0,
            f"Capital infusion Rs{infusion:,.0f}",
            meta['total_infused'])

    _save_tester_meta(name, meta)
    print(f"\n  ✅ Infusion complete. "
          f"Total infused: "
          f"Rs{meta['total_infused']:,.0f}")


# ── RESET TESTER ──────────────────────────────────────────────
def _reset_tester(name):
    confirm = input(
        f"\n  Reset '{name}'? "
        f"ALL data cleared. (yes/no): "
    ).strip().lower()
    if confirm != 'yes':
        print("  Cancelled.")
        return
    import shutil
    shutil.rmtree(_tester_dir(name), ignore_errors=True)
    print(f"  ✅ Tester '{name}' reset.")


# ── TESTER SUB-MENU ───────────────────────────────────────────
def _tester_sub_menu(name):
    TMENU = f"""
  ── TESTER: {name.upper()} {'─'*30}
  a. Run tester   (exits → redeploy → log)
  b. View positions  (live indicators + P&L + Nifty)
  c. View log        (transaction history)
  d. Performance     (P&L vs Nifty 50 + chart)
  e. Infuse capital  (pyramid / new stocks / split)
  f. Reset           (clear all data)
  g. Back
  {'─'*45}"""

    while True:
        meta    = _load_tester_meta(name)
        df_port = _load_tester_port(name)
        started = os.path.exists(_tester_meta_file(name))

        print(f"\n  Tester '{name}': "
              f"{'ACTIVE' if started else 'NOT STARTED'} | "
              f"Capital Rs{meta.get('capital',0):,.0f} | "
              f"{len(df_port)} positions | "
              f"Last run: {meta.get('last_run','never')}")
        print(TMENU)
        choice = input("  Choice (a-g): ").strip().lower()

        if choice == 'a':
            if not freshness_gate('run anyway'):
                continue
            if not started:
                _init_tester(name)
            else:
                _run_tester_update(name)

        elif choice == 'b':
            if not started:
                print("  Not started. Run (a) first.")
            else:
                _view_positions(name)

        elif choice == 'c':
            if not started:
                print("  Not started. Run (a) first.")
            else:
                try:
                    n = int(input(
                        "  Show last N transactions "
                        "[20]: ").strip() or '20')
                except:
                    n = 20
                _view_log(name, n)

        elif choice == 'd':
            if not started:
                print("  Not started. Run (a) first.")
            else:
                _view_performance(name)

        elif choice == 'e':
            if not started:
                print("  Not started. Run (a) first.")
            else:
                _infuse_capital(name)

        elif choice == 'f':
            _reset_tester(name)
            break

        elif choice == 'g':
            break

        else:
            print("  Invalid. Enter a-g.")


# ── MAIN AUTO-TESTER MENU ─────────────────────────────────────
def run_auto_tester():
    """
    Main entry point for Option 4.
    Supports multiple named testers.
    """
    while True:
        testers = _list_testers()

        print(f"\n  AUTO-TESTER — MULTIPLE PORTFOLIOS")
        print(f"  {'─'*45}")
        if testers:
            print(f"  Existing testers:")
            for i, t in enumerate(testers, 1):
                meta    = _load_tester_meta(t)
                df_port = _load_tester_port(t)
                cap     = meta.get('capital', 0)
                n_pos   = len(df_port)
                last_r  = meta.get('last_run', 'never')

                # Quick P&L
                total_inv  = sum(
                    float(r.get('Entry_Price',0)) *
                    int(r.get('Quantity',0))
                    for _, r in df_port.iterrows())
                total_curr = sum(
                    get_current_price(str(r.get('Symbol','')))
                    * int(r.get('Quantity',0))
                    for _, r in df_port.iterrows()) \
                    if n_pos > 0 else 0
                pl_pct = ((total_curr - total_inv) /
                          total_inv * 100
                          if total_inv > 0 else 0)

                print(f"    {i}. {t:<25} "
                      f"Rs{cap:>10,.0f}  "
                      f"{n_pos:>3}pos  "
                      f"P&L:{pl_pct:>+6.1f}%  "
                      f"Last:{last_r}")
            print(f"    {len(testers)+1}. "
                  f"Create new tester")
            print(f"    {len(testers)+2}. "
                  f"Back to main menu")
        else:
            print("  No testers yet.")
            print("    1. Create new tester")
            print("    2. Back to main menu")

        try:
            c = int(input(
                f"\n  Choice: ").strip())
        except:
            break

        n_options = len(testers) + 2
        if testers and 1 <= c <= len(testers):
            _tester_sub_menu(testers[c - 1])

        elif c == (len(testers) + 1):
            # Create new tester
            tname = input(
                "\n  New tester name "
                "(e.g. 'metals_apr26'): "
            ).strip().lower().replace(' ', '_')
            if not tname:
                print("  Invalid name.")
                continue
            if tname in testers:
                print(f"  '{tname}' already exists.")
                continue
            os.makedirs(_tester_dir(tname), exist_ok=True)
            _tester_sub_menu(tname)

        elif c == n_options:
            break
        else:
            print("  Invalid choice.")


# ── COMPLETION ────────────────────────────────────────────────
print("─" * 55)
print("✅  Cell 4 complete — Auto-tester engine loaded")
print("     run_auto_tester()      — main entry point")
print("     Multiple named testers supported")
print("     Sub-functions per tester:")
print("       _init_tester()         — first run")
print("       _run_tester_update()   — daily/data update")
print("       _view_positions()      — live indicators")
print("       _view_log()            — transaction history")
print("       _view_performance()    — P&L vs Nifty")
print("       _infuse_capital()      — pyramid/new/split")
print("       _reset_tester()        — clear and restart")
print()
print("  Now call run_optimizer_menu() to start.")

───────────────────────────────────────────────────────
✅  Cell 4 complete — Auto-tester engine loaded
     run_auto_tester()      — main entry point
     Multiple named testers supported
     Sub-functions per tester:
       _init_tester()         — first run
       _run_tester_update()   — daily/data update
       _view_positions()      — live indicators
       _view_log()            — transaction history
       _view_performance()    — P&L vs Nifty
       _infuse_capital()      — pyramid/new/split
       _reset_tester()        — clear and restart

  Now call run_optimizer_menu() to start.


In [16]:
# ============================================================
# DAY 15 — Portfolio Optimizer
# Cell 5 : Actual Portfolio — Build/Update (Option 5)
#           and Review (Option 6)
#
# Reads: long_term_portfolio.csv (from run_portfolio.py)
#        OR optimizer's own actual_portfolio.csv
# Shows: P&L + all flags + Nifty comparison + chart
#        + new candidates from fresh pipeline
#
# Weekly update flow:
#   1. Mark existing holdings to market
#   2. Ask free cash from exits + fresh infusion
#   3. Size new positions against TOTAL portfolio value
#   4. Show new candidates from fresh pipeline
# ============================================================

ACTUAL_PORT_FILE = os.path.join(
    PORTFOLIO_DIR, 'actual_portfolio.csv')
ACTUAL_META_FILE = os.path.join(
    OPT_STATE_DIR, 'actual_portfolio_meta.txt')

# ── ACTUAL PORTFOLIO HELPERS ──────────────────────────────────
def _load_actual_meta():
    meta = {
        'capital'       : 0.0,
        'total_infused' : 0.0,
        'n_stocks'      : DEFAULT_STOCK_COUNT,
        'start_date'    : today_str,
        'last_run'      : '',
        'last_run_ts'   : 0.0,
    }
    if not os.path.exists(ACTUAL_META_FILE):
        return meta
    try:
        with open(ACTUAL_META_FILE) as f:
            for line in f:
                line = line.strip()
                if '=' not in line:
                    continue
                k, v = line.split('=', 1)
                k, v = k.strip(), v.strip()
                if k == 'capital':
                    meta['capital'] = float(v)
                elif k == 'total_infused':
                    meta['total_infused'] = float(v)
                elif k == 'n_stocks':
                    meta['n_stocks'] = int(v)
                elif k == 'start_date':
                    meta['start_date'] = v
                elif k == 'last_run':
                    meta['last_run'] = v
                elif k == 'last_run_ts':
                    meta['last_run_ts'] = float(v)
    except:
        pass
    return meta

def _save_actual_meta(meta):
    with open(ACTUAL_META_FILE, 'w') as f:
        f.write(f"capital={meta['capital']}\n")
        f.write(f"total_infused="
                f"{meta.get('total_infused',0)}\n")
        f.write(f"n_stocks="
                f"{meta.get('n_stocks',DEFAULT_STOCK_COUNT)}\n")
        f.write(f"start_date={meta.get('start_date','')}\n")
        f.write(f"last_run={meta.get('last_run','')}\n")
        f.write(f"last_run_ts="
                f"{meta.get('last_run_ts',0)}\n")

def _load_actual_port():
    """
    Load actual portfolio.
    First tries optimizer's own actual_portfolio.csv.
    Falls back to run_portfolio.py's long_term_portfolio.csv.
    """
    if os.path.exists(ACTUAL_PORT_FILE):
        return load_port(ACTUAL_PORT_FILE, OPT_PORT_COLS)

    # Fallback — import from run_portfolio.py's file
    if os.path.exists(LT_PORT_FILE):
        df_lt = pd.read_csv(LT_PORT_FILE)
        print("  ℹ️  Loaded from long_term_portfolio.csv")
        print("     Columns will be mapped to optimizer format.")
        # Map run_portfolio columns to OPT_PORT_COLS
        df_out = pd.DataFrame(columns=OPT_PORT_COLS)
        col_map = {
            'Symbol'          : 'Symbol',
            'Entry_Price'     : 'Entry_Price',
            'Entry_Date'      : 'Entry_Date',
            'Quantity'        : 'Quantity',
            'Cap_Category'    : 'Cap_Category',
            'Invested'        : 'Invested',
            'Sector_Rank_At_Entry': 'Sector_Rank_At_Entry',
            'Cap_Rank_At_Entry'   : 'Cap_Rank_At_Entry',
        }
        for col_out, col_in in col_map.items():
            if col_in in df_lt.columns:
                df_out[col_out] = df_lt[col_in]
        # Fill missing optimizer columns
        for col in OPT_PORT_COLS:
            if col not in df_out.columns or \
               df_out[col].isna().all():
                df_out[col] = None
        return df_out

    return pd.DataFrame(columns=OPT_PORT_COLS)

def _save_actual_port(df):
    save_port(df, ACTUAL_PORT_FILE)


# ── OPTION 5 : BUILD / UPDATE ACTUAL PORTFOLIO ───────────────
def run_build_actual_portfolio():
    """
    Build or update the actual (real money) portfolio.

    First time: run fresh pipeline → choose stocks → save
    Subsequent: add/remove stocks manually + weekly update
                with free cash redeployment
    """
    print("\n" + "─"*55)
    print("  ACTUAL PORTFOLIO — BUILD / UPDATE")
    print("─"*55)

    df_port = _load_actual_port()
    meta    = _load_actual_meta()
    started = os.path.exists(ACTUAL_PORT_FILE) or \
              os.path.exists(LT_PORT_FILE)

    print(f"\n  Current holdings: {len(df_port)} stocks")
    if meta.get('capital', 0) > 0:
        print(f"  Capital: Rs{meta['capital']:,.0f}")
        print(f"  Started: {meta.get('start_date','?')}")

    print("\n  Options:")
    print("    1. Fresh start — run optimizer, "
          "select stocks")
    print("    2. Add stocks manually")
    print("    3. Remove stocks")
    print("    4. Edit price / quantity")
    print("    5. Weekly update — deploy free cash "
          "to new stocks")
    print("    6. Back")

    choice = input("\n  Choice (1-6): ").strip()

    # ── Option 1: Fresh start ─────────────────────────────
    if choice == '1':
        if len(df_port) > 0:
            confirm = input(
                f"  Portfolio has {len(df_port)} stocks. "
                f"Replace with fresh optimizer run? "
                f"(yes/no): "
            ).strip().lower()
            if confirm != 'yes':
                print("  Cancelled.")
                return

        if not freshness_gate('start anyway'):
            return

        df_all, df_final, status, capital, n_stocks = \
            run_pipeline(label='actual portfolio')

        if status in ('NO_STOCKS_L1', 'NO_BUY',
                      'NO_STOCKS_L4'):
            print("  No qualifying stocks. "
                  "Portfolio not changed.")
            return

        print_pipeline_table(
            df_final, capital,
            title='ACTUAL PORTFOLIO — OPTIMIZER SELECTION')

        confirm = input(
            "\n  Accept this allocation "
            "and save as actual portfolio? "
            "(yes/no): "
        ).strip().lower()

        if confirm != 'yes':
            print("  Cancelled. Nothing saved.")
            return

        # Build portfolio from allocated stocks
        allocated = df_final[
            df_final['Shares'] > 0
        ].copy() if 'Shares' in df_final.columns \
            else pd.DataFrame()

        # Exclude RETREATING
        allocated = allocated[
            ~allocated['Health_Label'].isin(
                ['RETREATING'])
        ].copy()

        port_records = []
        now_str      = datetime.now().strftime('%H:%M:%S')

        for _, row in allocated.iterrows():
            sym    = str(row['Symbol'])
            price  = float(
                row.get('Current Price', 0) or 0)
            shares = int(row.get('Shares', 0) or 0)
            inv    = float(row.get('Invested', 0) or 0)
            if shares == 0 or price <= 0:
                continue

            ml_pred, _ = get_current_ml(sym)
            trend      = get_current_trend(sym)
            ema50p     = compute_ema50_pct(sym)
            score      = float(
                row.get('Optimizer_Score', 0) or 0)
            h_lbl      = str(
                row.get('Health_Label', ''))
            stop_r     = float(
                row.get('Stop_Price_Ref', 0) or 0)
            rows_t = tech_df[tech_df['Symbol'] == sym]
            sec_rnk = float(rows_t.iloc[0].get(
                'Sector Score', 0) or 0) \
                if len(rows_t) > 0 else 0.0
            cap_rnk = float(rows_t.iloc[0].get(
                'Cap Score', 0) or 0) \
                if len(rows_t) > 0 else 0.0
            cap_cat = str(rows_t.iloc[0].get(
                'Cap Category', '')) \
                if len(rows_t) > 0 else ''

            port_records.append({
                'Symbol'                  : sym,
                'Entry_Price'             : price,
                'Entry_Date'              : today_str,
                'Quantity'                : shares,
                'Cap_Category'            : cap_cat,
                'Invested'                : inv,
                'Sector_Rank_At_Entry'    : sec_rnk,
                'Cap_Rank_At_Entry'       : cap_rnk,
                'ML_At_Entry'             : ml_pred,
                'Sector_Trend_At_Entry'   : trend,
                'EMA50_Pct_At_Entry'      : ema50p,
                'Optimizer_Score_At_Entry': score,
                'Health_At_Entry'         : h_lbl,
                'Stop_Price_Ref'          : stop_r,
                'Notes'                   : '',
            })

        df_port = pd.DataFrame(port_records)
        _save_actual_port(df_port)

        meta = {
            'capital'       : capital,
            'total_infused' : capital,
            'n_stocks'      : n_stocks,
            'start_date'    : today_str,
            'last_run'      : today_str,
            'last_run_ts'   : _time_mod.time(),
        }
        _save_actual_meta(meta)

        print(f"\n  ✅ Actual portfolio saved — "
              f"{len(port_records)} stocks  "
              f"Capital Rs{capital:,.0f}")

    # ── Option 2: Add manually ────────────────────────────
    elif choice == '2':
        df_port = add_stock_to_port(
            df_port, OPT_PORT_COLS)
        _save_actual_port(df_port)

    # ── Option 3: Remove ──────────────────────────────────
    elif choice == '3':
        df_port = remove_stocks_from_port(df_port)
        _save_actual_port(df_port)

    # ── Option 4: Edit ────────────────────────────────────
    elif choice == '4':
        df_port = update_quantity_price(df_port)
        _save_actual_port(df_port)

    # ── Option 5: Weekly update ───────────────────────────
    elif choice == '5':
        if len(df_port) == 0:
            print("  Portfolio is empty. "
                  "Use option 1 or 2 first.")
            return

        if not freshness_gate('run anyway'):
            return

        print(f"\n  WEEKLY UPDATE")
        print(f"  Current holdings: {len(df_port)} stocks")

        # Mark to market
        total_held_val = 0
        print(f"\n  Current P&L:")
        for _, row in df_port.iterrows():
            sym      = str(row['Symbol'])
            entry_px = float(
                row.get('Entry_Price', 0) or 0)
            qty      = int(row.get('Quantity', 0) or 0)
            curr_px  = get_current_price(sym)
            curr_px  = curr_px if curr_px > 0 else entry_px
            curr_val = curr_px * qty
            pl_pct   = ((curr_val - entry_px * qty) /
                        (entry_px * qty) * 100
                        if entry_px * qty > 0 else 0)
            total_held_val += curr_val

            curr_ml, _ = get_current_ml(sym)
            h          = compute_health(sym)
            ml_f       = ml_flag(
                curr_ml,
                str(row.get('ML_At_Entry',
                    'Bullish Continual')))
            l4_f       = ('🔴EXIT-L4'
                          if h['Health_Score'] <=
                          HEALTH_AUTO_EXIT_THRESHOLD
                          else '')
            flags      = ' '.join(
                f for f in [ml_f, l4_f] if f)

            print(f"  {'🔴' if flags else '✅'} "
                  f"{sym:<14} "
                  f"Rs{entry_px:.2f}→Rs{curr_px:.2f}  "
                  f"{pl_pct:+.1f}%  "
                  f"ML:{curr_ml[:14]}  "
                  f"Health:{h['Health_Label']}"
                  + (f"  {flags}" if flags else ""))

        # Ask free cash
        print(f"\n  Holdings market value: "
              f"Rs{total_held_val:,.0f}")
        try:
            exits_raw = input(
                "  Cash freed from exits this week "
                "(Rs, 0 if none): "
            ).strip().replace(',', '')
            exits_cash = float(exits_raw or '0')
        except:
            exits_cash = 0.0

        try:
            infuse_raw = input(
                "  Fresh capital to infuse "
                "(Rs, 0 if none): "
            ).strip().replace(',', '')
            infusion = float(infuse_raw or '0')
        except:
            infusion = 0.0

        free_cash       = exits_cash + infusion
        total_port_val  = total_held_val + free_cash
        n_stocks        = meta.get(
            'n_stocks', DEFAULT_STOCK_COUNT)
        target_pos_size = total_port_val / n_stocks
        slots_open      = max(0,
            n_stocks - len(df_port) +
            int(exits_cash / target_pos_size
                if target_pos_size > 0 else 0))

        print(f"\n  Free cash          : "
              f"Rs{free_cash:,.0f}")
        print(f"  Total portfolio    : "
              f"Rs{total_port_val:,.0f}")
        print(f"  Target per stock   : "
              f"Rs{target_pos_size:,.0f}")
        print(f"  Estimated slots    : {slots_open}")

        if free_cash < target_pos_size * 0.5:
            print(f"\n  ⚠️  Free cash Rs{free_cash:,.0f} "
                  f"< 50% of target "
                  f"Rs{target_pos_size:,.0f}.")
            print(f"     Minimum meaningful position: "
                  f"Rs{target_pos_size*0.5:,.0f}")
            ans = input(
                "  Proceed anyway? (y/n): "
            ).strip().lower()
            if ans != 'y':
                print("  Skipped. Add more capital "
                      "or wait for more exits.")
                return

        if free_cash > 0 and slots_open > 0:
            held_syms = set(df_port['Symbol'].tolist())
            print(f"\n  Finding new candidates "
                  f"(excluding {len(held_syms)} "
                  f"held stocks)...")

            _df_all, df_new, new_status, _c, _n = \
                run_pipeline(
                    label='actual portfolio update',
                    exclude_symbols=held_syms,
                    verbose=True,
                    capital=free_cash,
                    n_stocks=slots_open)

            if new_status not in (
                    'NO_STOCKS_L1', 'NO_BUY',
                    'NO_STOCKS_L4'):
                print_pipeline_table(
                    df_new, free_cash,
                    title='NEW CANDIDATES — '
                          'ACTUAL PORTFOLIO')

                confirm = input(
                    "\n  Accept these new positions? "
                    "(yes/no): "
                ).strip().lower()

                if confirm == 'yes':
                    buyable = df_new[
                        (df_new['Shares'] > 0) &
                        (~df_new['Health_Label'].isin(
                            ['RETREATING']))
                    ].copy() \
                        if 'Shares' in df_new.columns \
                        else pd.DataFrame()

                    new_records = []
                    for _, row in buyable.iterrows():
                        sym    = str(row['Symbol'])
                        price  = float(row.get(
                            'Current Price', 0) or 0)
                        shares = int(row.get(
                            'Shares', 0) or 0)
                        inv    = float(row.get(
                            'Invested', 0) or 0)
                        if shares == 0 or price <= 0:
                            continue

                        ml_pred, _ = get_current_ml(sym)
                        trend = get_current_trend(sym)
                        ema50p = compute_ema50_pct(sym)
                        score = float(row.get(
                            'Optimizer_Score', 0) or 0)
                        h_lbl = str(row.get(
                            'Health_Label', ''))
                        stop_r = float(row.get(
                            'Stop_Price_Ref', 0) or 0)
                        rows_t = tech_df[
                            tech_df['Symbol'] == sym]
                        sec_rnk = float(
                            rows_t.iloc[0].get(
                                'Sector Score', 0) or 0) \
                            if len(rows_t) > 0 else 0.0
                        cap_rnk = float(
                            rows_t.iloc[0].get(
                                'Cap Score', 0) or 0) \
                            if len(rows_t) > 0 else 0.0
                        cap_cat = str(
                            rows_t.iloc[0].get(
                                'Cap Category', '')) \
                            if len(rows_t) > 0 else ''

                        new_records.append({
                            'Symbol'            : sym,
                            'Entry_Price'       : price,
                            'Entry_Date'        : today_str,
                            'Quantity'          : shares,
                            'Cap_Category'      : cap_cat,
                            'Invested'          : inv,
                            'Sector_Rank_At_Entry': sec_rnk,
                            'Cap_Rank_At_Entry' : cap_rnk,
                            'ML_At_Entry'       : ml_pred,
                            'Sector_Trend_At_Entry': trend,
                            'EMA50_Pct_At_Entry': ema50p,
                            'Optimizer_Score_At_Entry': score,
                            'Health_At_Entry'   : h_lbl,
                            'Stop_Price_Ref'    : stop_r,
                            'Notes'             : '',
                        })

                    if new_records:
                        df_port = pd.concat(
                            [df_port,
                             pd.DataFrame(new_records)],
                            ignore_index=True)
                        _save_actual_port(df_port)
                        print(f"  ✅ Added "
                              f"{len(new_records)} "
                              f"new stocks.")
            else:
                print("  No new qualifying stocks.")

        # Update meta
        if infusion > 0:
            meta['total_infused'] = (
                meta.get('total_infused',
                         meta.get('capital', 0))
                + infusion)
        meta['last_run']    = today_str
        meta['last_run_ts'] = _time_mod.time()
        _save_actual_meta(meta)

    elif choice == '6':
        return
    else:
        print("  Invalid choice.")


# ── OPTION 6 : REVIEW ACTUAL PORTFOLIO ───────────────────────
def run_review_actual_portfolio():
    """
    Full review of actual portfolio:
      - P&L per stock + total
      - All flags: EXIT / REDUCE / MONITOR / STOP / CORP / LOCK
      - Nifty 50 comparison + alpha
      - New candidates from fresh pipeline
      - Chart (optional)
    """
    print("\n" + "─"*55)
    print("  ACTUAL PORTFOLIO — REVIEW")
    print("─"*55)

    df_port = _load_actual_port()
    meta    = _load_actual_meta()

    if len(df_port) == 0:
        print("  No holdings. Use Option 5 to build "
              "your portfolio first.")
        return

    capital    = meta.get('capital', 0)
    start_date = meta.get('start_date', today_str)

    lines = []
    def p(line=''): lines.append(str(line))

    p("=" * 80)
    p("  ACTUAL PORTFOLIO REVIEW")
    p(f"  Generated : "
      f"{datetime.now().strftime('%Y-%m-%d %H:%M')}")
    p(f"  Capital   : Rs{capital:,.0f}  "
      f"| Started: {start_date}")
    p("=" * 80)

    total_inv  = 0
    total_curr = 0
    exit_count = 0
    flag_lines = []

    p(f"\n  {'Symbol':<14} {'Entry':>8} {'Curr':>8} "
      f"{'P&L%':>6}  "
      f"{'ML':<20} {'Health':<10} "
      f"{'EMA50%':>7}  Flags")
    p("  " + "─"*90)

    for _, row in df_port.iterrows():
        sym      = str(row['Symbol'])
        entry_px = float(row.get('Entry_Price', 0) or 0)
        qty      = int(row.get('Quantity', 0) or 0)
        invested = entry_px * qty
        total_inv += invested

        curr_px  = get_current_price(sym)
        curr_px  = curr_px if curr_px > 0 else entry_px
        curr_val = curr_px * qty
        total_curr += curr_val
        pl_pct   = ((curr_val - invested) / invested * 100
                    if invested > 0 else 0)

        curr_ml, curr_conf = get_current_ml(sym)
        ml_entry    = str(row.get(
            'ML_At_Entry', 'Bullish Continual'))
        trend_entry = str(row.get(
            'Sector_Trend_At_Entry', ''))
        curr_trend  = get_current_trend(sym)
        curr_ema50  = compute_ema50_pct(sym)
        stop_ref    = float(
            row.get('Stop_Price_Ref', 0) or 0)
        h           = compute_health(sym)
        h_lbl       = h['Health_Label']
        h_scr       = h['Health_Score']

        # All flags
        f1 = ml_flag(curr_ml, ml_entry)
        f2 = trend_flag(curr_trend, trend_entry)
        f3 = ema50_monitor_flag(sym)
        f4 = stop_hit_flag(curr_px, stop_ref)
        f5 = corp_action_flag(sym, entry_px)
        f6 = gains_lock_flag(curr_ml, entry_px, curr_px)
        f7 = ('🔴EXIT-L4'
              if h_scr <= HEALTH_AUTO_EXIT_THRESHOLD
              else '')

        all_flags = ' '.join(
            f for f in [f1,f2,f3,f4,f5,f6,f7] if f)

        if f1 or f7:
            exit_count += 1

        h_str = (f"{HEALTH_ICON.get(h_lbl,'?')}"
                 f"({h_scr})")

        p(f"  {sym:<14} "
          f"Rs{entry_px:>7.2f} "
          f"Rs{curr_px:>7.2f} "
          f"{pl_pct:>+5.1f}%  "
          f"{curr_ml[:18]:<20} "
          f"{h_str:<10} "
          f"{curr_ema50:>+6.1f}%"
          + (f"  {all_flags}" if all_flags else ""))

        if all_flags:
            flag_lines.append(
                f"  ⚡ {sym}: {all_flags}")

    # Portfolio totals
    port_pl     = total_curr - total_inv
    port_pl_pct = (port_pl / total_inv * 100
                   if total_inv > 0 else 0)

    p("\n  " + "─"*80)
    p(f"  Stocks    : {len(df_port)}")
    p(f"  Invested  : Rs{total_inv:,.0f}")
    p(f"  Current   : Rs{total_curr:,.0f}")
    p(f"  P&L       : Rs{port_pl:+,.0f} "
      f"({port_pl_pct:+.1f}%)")

    # Nifty comparison
    nifty = fetch_nifty_return(start_date)
    if nifty:
        n_s, n_c, n_p = nifty
        alpha = port_pl_pct - n_p
        p(f"  Nifty 50  : {n_p:+.1f}%  "
          f"({n_s:.0f}→{n_c:.0f})")
        p(f"  Alpha     : {alpha:+.1f}%  "
          f"{'✅ Outperforming' if alpha > 0 else '❌ Underperforming'}")

    # Action summary
    if flag_lines:
        p(f"\n  ── ACTION REQUIRED ({exit_count} exits) ──")
        for fl in flag_lines:
            p(fl)

    # New candidates from fresh pipeline
    p(f"\n  ── NEW CANDIDATES (not in portfolio) ──")
    held_syms = set(df_port['Symbol'].tolist())

    _df_new, _, new_status, _, _ = run_pipeline(
        label='review — new candidates',
        exclude_symbols=held_syms,
        verbose=False,
        capital=capital,
        n_stocks=5)

    if (new_status not in ('NO_STOCKS_L1', 'NO_BUY',
                           'NO_STOCKS_L4')
            and len(_df_new) > 0):
        investable = _df_new[
            ~_df_new['Health_Label'].isin(
                ['RETREATING'])
        ].head(8)

        if len(investable) > 0:
            p(f"  Top new qualifying stocks "
              f"(not currently held):")
            p(f"  {'Symbol':<14} {'Score':>5}  "
              f"{'Health':<10} {'EMA50%':>7}  "
              f"{'Trend':<14}  R20")
            p("  " + "─"*65)
            for _, r in investable.iterrows():
                h   = compute_health(str(r['Symbol']))
                det = h['Health_Detail']
                r20 = [x for x in det.split('  ')
                       if x.startswith('R20:')]
                r20_s = r20[0] if r20 else '—'
                p(f"  {str(r['Symbol']):<14} "
                  f"{float(r.get('Optimizer_Score',0)):>5.1f}  "
                  f"{HEALTH_ICON.get(h['Health_Label'],'?'):<10} "
                  f"{float(r.get('EMA50_Pct',0)):>+6.1f}%  "
                  f"{str(r.get('Sector Trend','')):<14}  "
                  f"{r20_s}")
        else:
            p("  No new qualifying stocks right now.")
    else:
        p("  No new qualifying stocks right now.")

    # Print and save
    for line in lines:
        print(line)

    report_path = os.path.join(
        REPORT_DIR,
        f'actual_portfolio_review_{today_file}.txt')
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"\n  Report saved → "
          f"{os.path.basename(report_path)}")

    # Chart
    if input(
            "\n  Generate chart vs Nifty 50? (y/n): "
    ).strip().lower() == 'y':
        plot_portfolio_vs_nifty(
            'actual_portfolio', df_port, start_date)

    # Update meta timestamp
    meta['last_run']    = today_str
    meta['last_run_ts'] = _time_mod.time()
    _save_actual_meta(meta)


# ── WIRE INTO MENU ────────────────────────────────────────────
print("─" * 55)
print("✅  Cell 5 complete — Actual portfolio loaded")
print("     run_build_actual_portfolio()   — Option 5")
print("     run_review_actual_portfolio()  — Option 6")
print()
print("  Update Cell 2 run_optimizer_menu():")
print("    choice '5' → run_build_actual_portfolio()")
print("    choice '6' → run_review_actual_portfolio()")

───────────────────────────────────────────────────────
✅  Cell 5 complete — Actual portfolio loaded
     run_build_actual_portfolio()   — Option 5
     run_review_actual_portfolio()  — Option 6

  Update Cell 2 run_optimizer_menu():
    choice '5' → run_build_actual_portfolio()
    choice '6' → run_review_actual_portfolio()


In [17]:
# ============================================================
# DAY 15 — Portfolio Optimizer
# Cell 6 : State save + standalone script assembly
#
# Two jobs:
#   1. save_optimizer_state() — saves current week's
#      qualifying stocks snapshot to optimizer_state.csv
#      Used next week to detect new qualifying stocks
#
#   2. Assembles all cells into run_portfolio_optimizer.py
#      so you can run it directly after run_weekly.py
#      without opening Jupyter
# ============================================================

# ── JOB 1 : SAVE OPTIMIZER STATE ─────────────────────────────
def save_optimizer_state():
    """
    Save snapshot of current qualifying stocks.
    Compared next run to detect:
      - New stocks that just entered qualifying list
      - Stocks that dropped out (useful for watchlist)
      - Score changes (rank movers)

    Called automatically at end of every pipeline run.
    File: data/portfolio/optimizer/optimizer_state.csv
    """
    try:
        df_f = filter_candidates(tech_df, verbose=False)

        if len(df_f) == 0:
            print("  ℹ️  No qualifying stocks — "
                  "state not updated.")
            return

        df_f['Optimizer_Score'] = df_f.apply(
            compute_optimizer_score, axis=1)
        df_f = apply_health_check(df_f)

        # Columns to save
        state_cols = [
            'Symbol', 'Sector', 'Cap Category',
            'Current Price', 'EMA50_Pct',
            'ML_Prediction', 'ML_Confidence_f',
            'Sector Trend', 'Tech Score_f',
            'Optimizer_Score', 'Health_Label',
            'Health_Score', 'Tier',
        ]
        df_state = df_f[[
            c for c in state_cols
            if c in df_f.columns
        ]].copy()
        df_state['Snapshot_Date'] = today_str
        df_state['Snapshot_Time'] = \
            datetime.now().strftime('%H:%M:%S')

        df_state.to_csv(STATE_FILE, index=False)
        print(f"  ✅ State saved — "
              f"{len(df_state)} stocks → "
              f"{os.path.basename(STATE_FILE)}")

    except Exception as e:
        print(f"  ⚠️  State save failed: {e}")


def check_new_qualifiers():
    """
    Compare current qualifying stocks against last saved state.
    Reports stocks that newly entered the qualifying list.
    Called at start of each pipeline run for awareness.
    """
    if prev_state is None or len(prev_state) == 0:
        print("  ℹ️  No previous state — "
              "first run, no comparison.")
        return

    try:
        df_curr = filter_candidates(
            tech_df, verbose=False)
        if len(df_curr) == 0:
            return

        prev_syms = set(
            prev_state['Symbol'].tolist())
        curr_syms = set(df_curr['Symbol'].tolist())

        new_syms     = curr_syms - prev_syms
        dropped_syms = prev_syms - curr_syms

        if new_syms:
            print(f"\n  🆕 NEW qualifiers since last run "
                  f"({len(new_syms)} stocks):")
            df_new_rows = df_curr[
                df_curr['Symbol'].isin(new_syms)
            ].copy()
            df_new_rows['Optimizer_Score'] = \
                df_new_rows.apply(
                    compute_optimizer_score, axis=1)
            df_new_rows = df_new_rows.sort_values(
                'Optimizer_Score', ascending=False)

            for _, r in df_new_rows.iterrows():
                h = compute_health(str(r['Symbol']))
                print(f"     {str(r['Symbol']):<14} "
                      f"Score:{float(r.get('Optimizer_Score',0)):.1f}  "
                      f"Health:{h['Health_Label']}  "
                      f"EMA50%:{float(r.get('EMA50_Pct',0)):+.1f}%  "
                      f"{str(r.get('Sector Trend',''))}")

        if dropped_syms:
            print(f"\n  📉 DROPPED from qualifying "
                  f"({len(dropped_syms)} stocks): "
                  f"{', '.join(sorted(dropped_syms))}")

        if not new_syms and not dropped_syms:
            print("  ℹ️  Same qualifying stocks as "
                  "last run — no changes.")

    except Exception as e:
        print(f"  ⚠️  Comparison failed: {e}")


# ── JOB 2 : ASSEMBLE STANDALONE SCRIPT ───────────────────────
def build_standalone_script():
    """
    Combine all notebook cells into a single
    run_portfolio_optimizer.py script.

    The script can be run directly:
      python run_portfolio_optimizer.py

    Or triggered automatically from run_weekly.py:
      subprocess.run(['python',
                      'run_portfolio_optimizer.py'])
    """
    script_path = os.path.join(
        BASE_DIR, 'run_portfolio_optimizer.py')

    header = '''#!/usr/bin/env python3
# ============================================================
# run_portfolio_optimizer.py
# AI Stock Screener — Portfolio Optimizer (Day 15)
#
# Run after run_weekly.py:
#   python run_portfolio_optimizer.py
#
# Or add to end of run_weekly.py:
#   import subprocess
#   subprocess.Popen(['python',
#       r'E:\\...\\run_portfolio_optimizer.py'])
#
# Generated from day15_portfolio_optimizer.ipynb
# ============================================================

'''

    # Collect all cell source files if they exist
    # Otherwise write instruction to export from Jupyter
    instructions = f'''
# ── HOW TO GENERATE THIS SCRIPT ──────────────────────────────
#
# Option A — Export from Jupyter (recommended):
#   File → Download as → Python (.py)
#   Then remove the cell markers (# In[N]:)
#
# Option B — The cells in order are:
#   Cell 1 : Imports, paths, constants
#   Cell 2 : Data load, helpers, menu
#   Cell 3 : 4-layer pipeline engine
#   Cell 4 : Auto-tester engine
#   Cell 5 : Actual portfolio
#   Cell 6 : State save (this file)
#
# Entry point (add at bottom after all cells):

if __name__ == '__main__':
    import sys
    if '--auto' in sys.argv:
        # Called automatically from run_weekly.py
        # Run auto-tester update on all active testers
        print("\\nAuto-tester update triggered by run_weekly.py")
        testers = _list_testers()
        if testers:
            for tname in testers:
                print(f"\\nUpdating tester: {{tname}}")
                _run_tester_update(tname)
        else:
            print("No active testers.")
        save_optimizer_state()
        check_new_qualifiers()
    else:
        # Interactive mode — show menu
        run_optimizer_menu()
'''

    main_block = '''

# ── ENTRY POINT ───────────────────────────────────────────────
if __name__ == '__main__':
    import sys

    # Save state on every run
    save_optimizer_state()
    check_new_qualifiers()

    if '--auto' in sys.argv:
        # Triggered by run_weekly.py — update all testers
        print("\\nAuto-update: checking all testers...")
        testers = _list_testers()
        if testers:
            for tname in testers:
                print(f"\\n── Tester: {tname} ──")
                _run_tester_update(tname)
        else:
            print("No active testers.")
        print("\\nAuto-update complete.")
    else:
        # Interactive — show menu
        run_optimizer_menu()
'''

    # Write the script with instructions
    with open(script_path, 'w', encoding='utf-8') as f:
        f.write(header)
        f.write("# " + "─"*60 + "\n")
        f.write("# INSTRUCTIONS TO COMPLETE THIS FILE\n")
        f.write("# " + "─"*60 + "\n")
        f.write(instructions)
        f.write("\n# " + "─"*60 + "\n")
        f.write("# Add the __main__ block below after\n")
        f.write("# pasting all cell contents above it:\n")
        f.write("# " + "─"*60 + "\n")
        f.write(main_block)

    print(f"  ✅ Script template written → "
          f"{os.path.basename(script_path)}")
    print(f"  Open it and paste cell contents "
          f"in order (Cell 1 → Cell 5)")
    print(f"  The __main__ block is already at the bottom")

    return script_path


# ── ADD TO run_weekly.py ──────────────────────────────────────
def show_weekly_integration():
    """
    Show the exact lines to add to run_weekly.py
    so the optimizer runs automatically after weekly update.
    """
    code = f'''
# ── ADD THESE LINES AT THE END OF run_weekly.py ──────────────
# Triggers portfolio optimizer automatically after weekly run

import subprocess, os

optimizer_script = r'{os.path.join(BASE_DIR, "run_portfolio_optimizer.py")}'

if os.path.exists(optimizer_script):
    print("\\nTriggering portfolio optimizer...")
    subprocess.Popen(
        ['python', optimizer_script, '--auto'],
        creationflags=subprocess.CREATE_NEW_CONSOLE
    )
    print("Optimizer running in background.")
# ─────────────────────────────────────────────────────────────
'''
    print(code)


# ── RUN BOTH JOBS ─────────────────────────────────────────────
print("─" * 55)
print("  Cell 6 — State save + Script assembly")
print("─" * 55)

print("\n  Job 1: Saving optimizer state...")
save_optimizer_state()

print("\n  Job 2: Building standalone script template...")
script_path = build_standalone_script()

print(f"\n  Job 3: Integration code for run_weekly.py")
show_weekly_integration()

print(f"\n{'═'*55}")
print(f"✅  ALL CELLS COMPLETE — Day 15 Portfolio Optimizer")
print(f"{'═'*55}")
print(f"""
  Summary of what was built:

  MENU OPTIONS:
    1. Fresh recommendations    → 4-layer pipeline
    2. Create/manage model portfolio
    3. Review model portfolio   → P&L + flags + Nifty
    4. Auto-tester              → autonomous paper trading
    5. Build/update actual portfolio
    6. Review actual portfolio  → P&L + flags + new candidates

  4-LAYER PIPELINE:
    Layer 1: 5 gates + Tier 3 block (→ ~23-53 stocks)
    Layer 2: Score with EMA50%, Tech, ML, Ranks, F25d,
             sector mult, Tier mult, S6 mult (20d range)
    Layer 3: Equal allocation, dynamic sector cap
    Layer 4: 6 signals including S6 (20d range upside room)

  KEY DESIGN DECISIONS:
    Equal allocation (not ATR weighted) — validated by test
    ML Bull Cont continuity = primary exit signal
    Layer 4 MIXED = valid entry (14/23 stocks are MIXED)
    S6 bonus: 20d range 40-75% → 1.08× score multiplier
    Fresh pipeline always — never stale watchlist
    Multiple named auto-testers supported

  RUN ORDER:
    run_weekly.py → run_portfolio_optimizer.py
    (or add auto-trigger to end of run_weekly.py)

  NEXT STEPS:
    1. Export notebook as .py
    2. Test Option 4 auto-tester with a small capital
    3. After 2-3 weeks compare tester vs Nifty
    4. Add --auto trigger to run_weekly.py
""")

───────────────────────────────────────────────────────
  Cell 6 — State save + Script assembly
───────────────────────────────────────────────────────

  Job 1: Saving optimizer state...
  ✅ State saved — 23 stocks → optimizer_state.csv

  Job 2: Building standalone script template...
  ✅ Script template written → run_portfolio_optimizer.py
  Open it and paste cell contents in order (Cell 1 → Cell 5)
  The __main__ block is already at the bottom

  Job 3: Integration code for run_weekly.py

# ── ADD THESE LINES AT THE END OF run_weekly.py ──────────────
# Triggers portfolio optimizer automatically after weekly run

import subprocess, os

optimizer_script = r'E:\learn\Project 1 AI Screener\stock-ai-india\run_portfolio_optimizer.py'

if os.path.exists(optimizer_script):
    print("\nTriggering portfolio optimizer...")
    subprocess.Popen(
        ['python', optimizer_script, '--auto'],
        creationflags=subprocess.CREATE_NEW_CONSOLE
    )
    print("Optimizer running in background.

In [18]:
# ── UPDATED DIAGNOSTIC — confirm S7 on key stocks ─────────────
print("=== LAYER 4 with S7 — key stocks ===\n")
print(f"{'Symbol':<14} {'Scr':>4}  {'Health':<11} "
      f"{'S7':>4}  Detail")
print("─"*90)

for sym in ['GMDCLTD','TORNTPOWER','SARDAEN',
            'NAVA','GALLANTT','RAMRAT',
            'PARAS','SIEMENS']:
    rows = tech_df[tech_df['Symbol'] == sym]
    if len(rows) == 0:
        continue
    rows['EMA50_Pct'] = rows.apply(
        lambda r: (float(r.get('Current Price',0) or 0) -
                   float(r.get('EMA50',0) or 0)) /
                  float(r.get('EMA50',1) or 1) * 100,
        axis=1)
    rows['ML_Confidence_f'] = pd.to_numeric(
        rows['ML_Confidence'], errors='coerce').fillna(0)
    rows['Tech Score_f'] = pd.to_numeric(
        rows['Tech Score'], errors='coerce').fillna(0)
    score = compute_optimizer_score(rows.iloc[0])
    h     = compute_health(sym)
    s7    = h.get('S7_MACD', 0)
    s7_s  = ('+1↑' if s7 == 1
             else '-2↓↓' if s7 == -2
             else ' 0→')
    icon  = {'RISING':'✅','MIXED':'⚠️ ',
             'NEUTRAL':'⚠️ ','RETREATING':'🚫'}.get(
        h['Health_Label'],'❓')
    print(f"  {sym:<14} {h['Health_Score']:>4}  "
          f"{icon}{h['Health_Label']:<9} "
          f"{s7_s:>5}  {h['Health_Detail']}")

print("\n=== NAVA specifically — S7 detail ===")
h = compute_health('NAVA')
print(f"  Health     : {h['Health_Label']} "
      f"(score={h['Health_Score']})")
print(f"  S7_MACD    : {h.get('S7_MACD',0)}")
print(f"  Full detail: {h['Health_Detail']}")
print()
print("Expected: NAVA S7=-2 (MACD hist falling + price falling)")
print("Expected: NAVA health drops from MIXED(+3) to MIXED(+1)")
print("         one more bad day → NEUTRAL(0)")
print("         two more → RETREATING(-1) → excluded")

=== LAYER 4 with S7 — key stocks ===

Symbol          Scr  Health        S7  Detail
──────────────────────────────────────────────────────────────────────────────────────────
  GMDCLTD           5  ✅RISING      +1↑  Mom:+9.9%↑  R5:82%↑  E20:+4.25%↑  RSI:67→69↑  R20:83%→  MACD↑(+1.2)
  TORNTPOWER        0  ⚠️ NEUTRAL    -2↓↓  Mom:+2.1%↑  R5:44%→  E20:+4.33%↑  RSI:74→68↓  Rec✓  R20:84%→  MACD↓P↓(-10.7)
  SARDAEN           1  ⚠️ MIXED      -2↓↓  Mom:+1.7%↑  R5:26%↓  E20:+2.18%↑  RSI:59→  Rec✓  R20:73%✓  MACD↓P↓(-2.3)
  NAVA              1  ⚠️ MIXED      -2↓↓  Mom:+0.8%↑  R5:26%↓  E20:+1.94%↑  RSI:60→  Rec✓  R20:62%✓  MACD↓P↓(-3.6)
  GALLANTT          3  ⚠️ MIXED        0→  Mom:+4.0%↑  R5:51%→  E20:+5.58%↑  RSI:68→68↑  R20:79%→  MACD→
  RAMRAT            3  ⚠️ MIXED        0→  Mom:+15.6%↑  R5:91%↑  E20:+7.99%↑  RSI:OB↓78  Rec✓  R20:96%→  MACD→
  PARAS            -3  🚫RETREATING  -2↓↓  Mom:-0.8%↓  R5:24%↓  E20:+3.20%↑  RSI:62→  R20:77%→  MACD↓P↓(-5.1)
  SIEMENS          -3  🚫RETREATING  -2↓